# Assignment 04 — Bài toán 1: Chẩn đoán Đái tháo đường bằng CNN 1D

**Môn học:** Intelligent System Development — TS. Trần Đình Quế
**Sinh viên:** Đinh Hải Triều — B23DCCN843 — Lớp 06

---

## Mục tiêu

1. Cài đặt **mạng nơ-ron tích chập 1D (CNN)** hoàn toàn bằng **NumPy from
   scratch**: tự viết `conv1d`, `maxpool1d`, lan truyền ngược và Adam.
2. Dựng **hai bản tương đương bằng PyTorch và TensorFlow/Keras**, huấn luyện
   trong điều kiện được kiểm soát để so sánh ba nền tảng một cách công bằng.
3. Đối sánh CNN với **MLP 5 tầng của Assignment 03** và các mô hình **Học máy
   truyền thống**, trên cùng một tập kiểm thử.
4. **Kiểm chứng tính đúng đắn** của bản tự cài: tích chập khớp định nghĩa toán
   học, gradient viết tay khớp gradient sai phân số.
5. Làm rõ **Cảnh báo Mô hình hoá** ở slide 8: dữ liệu bảng KHÔNG có tính cục bộ
   không gian như ảnh. Ta kiểm chứng bằng thí nghiệm hoán vị thứ tự cột.
6. **Xuất trọng số ra `model_cnn.json`**, đẩy lên GitHub và gọi từ frontend để
   chạy serverless trên Vercel.

**Bài toán:** Binary Classification — dự đoán `Outcome` ∈ {0, 1} từ 8 chỉ số
lâm sàng, xem 8 đặc trưng đó như một chuỗi 1D độ dài 8 với 1 kênh.

In [1]:
# ============================================================================
# KHỐI 1 — Nạp thư viện, cố định seed và cấu hình hình vẽ
# matplotlib.use(Agg) cho phép chạy notebook cả khi không có giao diện đồ hoạ;
# mọi hình được lưu vào figures/ để chèn thẳng vào báo cáo.
# SEED = 42 cố định mọi nguồn ngẫu nhiên nên kết quả tái lập được giữa các lần chạy.
# ============================================================================
import json
import os
import pathlib
import time

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")   # tắt log khởi động của TF
os.environ.setdefault("TF_ENABLE_ONEDNN_OPTS", "0")  # bỏ oneDNN để số học ổn định

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report,
)

SEED = 42
np.random.seed(SEED)

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 150,
    "font.family": "DejaVu Sans",
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})
sns.set_palette("deep")

ROOT = pathlib.Path.cwd()
FIG = ROOT / "figures"
FIG.mkdir(exist_ok=True)
print("Thư mục làm việc :", ROOT)
print("Thư mục hình vẽ  :", FIG)

Thư mục làm việc : C:\Users\admin\Downloads\bt-thay quế\tuan 1\website_chandoan_tieu_duong
Thư mục hình vẽ  : C:\Users\admin\Downloads\bt-thay quế\tuan 1\website_chandoan_tieu_duong\figures


## 1. Nạp dữ liệu và khảo sát ban đầu (EDA)

Bộ dữ liệu **Pima Indians Diabetes**: 768 bệnh nhân nữ gốc Pima, 8 đặc trưng
lâm sàng, nhãn nhị phân `Outcome`.

In [2]:
# ============================================================================
# KHỐI 2 — Nạp dữ liệu và xem phân bố nhãn
# ============================================================================
df = pd.read_csv(ROOT / "data" / "pima_diabetes.csv")
print("Kích thước:", df.shape)
print("\nPhân bố nhãn:")
print(df["Outcome"].value_counts().rename({0: "Không tiểu đường", 1: "Tiểu đường"}))
print("\nTỉ lệ dương tính: %.2f%%" % (100 * df["Outcome"].mean()))
df.head()

Kích thước: (768, 9)

Phân bố nhãn:
Outcome
Không tiểu đường    500
Tiểu đường          268
Name: count, dtype: int64

Tỉ lệ dương tính: 34.90%


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [3]:
df.describe().T[["mean", "std", "min", "50%", "max"]].round(2)

,mean,std,min,50%,max
Pregnancies,3.85,3.37,0.00,3.00,17.00
Glucose,120.89,31.97,0.00,117.00,199.00
BloodPressure,69.11,19.36,0.00,72.00,122.00
SkinThickness,20.54,15.95,0.00,23.00,99.00
Insulin,79.80,115.24,0.00,30.50,846.00
BMI,31.99,7.88,0.00,32.00,67.10
DiabetesPedigreeFunction,0.47,0.33,0.08,0.37,2.42
Age,33.24,11.76,21.00,29.00,81.00
Outcome,0.35,0.48,0.00,0.00,1.00


### 1.1. Phát hiện giá trị thiếu bị mã hoá thành 0

Trong bộ Pima, giá trị `0` ở các cột sinh lý là **bất khả thi về mặt y học**
(không ai có Glucose = 0 hay BMI = 0). Đây thực chất là **missing value bị mã
hoá nhầm** — nếu để nguyên, mạng sẽ học phải các mẫu giả.

In [4]:
# ============================================================================
# KHỐI 3 — Đếm giá trị 0 bất khả thi ở các cột sinh lý
# ============================================================================
ZERO_AS_NAN = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
miss = pd.DataFrame({
    "Số giá trị 0": [(df[c] == 0).sum() for c in ZERO_AS_NAN],
    "Tỉ lệ (%)": [round(100 * (df[c] == 0).mean(), 2) for c in ZERO_AS_NAN],
}, index=ZERO_AS_NAN)
miss

,Số giá trị 0,Tỉ lệ (%)
Glucose,5,0.65
BloodPressure,35,4.56
SkinThickness,227,29.56
Insulin,374,48.70
BMI,11,1.43


In [5]:
# ============================================================================
# KHỐI 4 — Hình 1: bốn góc nhìn EDA
# (a) phân bố nhãn, (b) Glucose theo nhãn, (c) tỉ lệ giá trị 0, (d) tương quan.
# ============================================================================
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

counts = df["Outcome"].value_counts().sort_index()
axes[0, 0].bar(["Không tiểu đường (0)", "Tiểu đường (1)"], counts.values,
               color=["#3b82f6", "#ef4444"], width=0.55)
for i, v in enumerate(counts.values):
    axes[0, 0].text(i, v + 8, f"{v}\n({100*v/len(df):.1f}%)", ha="center", fontweight="bold")
axes[0, 0].set_title("(a) Phân bố nhãn — mất cân bằng nhẹ 65/35", fontweight="bold")
axes[0, 0].set_ylabel("Số bệnh nhân")
axes[0, 0].set_ylim(0, 620)

for lbl, color, name in [(0, "#3b82f6", "Không tiểu đường"), (1, "#ef4444", "Tiểu đường")]:
    sub = df.loc[df["Outcome"] == lbl, "Glucose"].replace(0, np.nan).dropna()
    axes[0, 1].hist(sub, bins=28, alpha=0.6, color=color, label=name)
axes[0, 1].set_title("(b) Glucose — đặc trưng phân tách mạnh nhất", fontweight="bold")
axes[0, 1].set_xlabel("Glucose (mg/dL)")
axes[0, 1].set_ylabel("Tần suất")
axes[0, 1].legend()

axes[1, 0].barh(miss.index, miss["Tỉ lệ (%)"], color="#f59e0b")
for i, v in enumerate(miss["Tỉ lệ (%)"]):
    axes[1, 0].text(v + 0.6, i, f"{v}%", va="center", fontweight="bold")
axes[1, 0].set_title("(c) Tỉ lệ giá trị 0 bất khả thi (missing trá hình)", fontweight="bold")
axes[1, 0].set_xlabel("% số mẫu")
axes[1, 0].set_xlim(0, 55)

corr = df.corr(numeric_only=True)
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            ax=axes[1, 1], cbar_kws={"shrink": 0.8}, annot_kws={"size": 7})
axes[1, 1].set_title("(d) Ma trận tương quan Pearson", fontweight="bold")
axes[1, 1].grid(False)

plt.tight_layout()
plt.savefig(FIG / "c1_fig1_eda.png", bbox_inches="tight")
plt.show()

C:\Users\admin\AppData\Local\Temp\ipykernel_23616\871872191.py:39: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 1.2. Tám cột này có "tính cục bộ" như điểm ảnh không?

Đây là câu hỏi trung tâm của slide 8. Với ảnh, hai điểm ảnh cạnh nhau **thực sự**
liên quan; kernel 3×3 vì thế có nghĩa. Với bảng, thứ tự cột do người tạo file CSV
quyết định: `Glucose` nằm cạnh `BloodPressure` **không** hàm ý quan hệ không gian.

Hình dưới minh hoạ: nếu tính cục bộ là thật thì các cặp cột **liền kề** phải
tương quan mạnh hơn các cặp **cách xa**. Ta vẽ |tương quan| theo khoảng cách
giữa hai cột để kiểm tra.

In [6]:
# ============================================================================
# KHỐI 5 — Hình 2: kiểm tra giả thiết "cột liền kề thì liên quan"
# Với ảnh, |corr| giảm rõ theo khoảng cách điểm ảnh. Nếu ở đây đường nằm ngang
# thì tính cục bộ không tồn tại, và đó chính là nội dung Cảnh báo Mô hình hoá.
# ============================================================================
feat_cols = [c for c in df.columns if c != "Outcome"]
FEATURES_PREVIEW = feat_cols
corr_abs = df[feat_cols].corr().abs().values
n_f = len(feat_cols)

dist_vals = {}
for i in range(n_f):
    for j in range(i + 1, n_f):
        dist_vals.setdefault(j - i, []).append(corr_abs[i, j])

dists = sorted(dist_vals)
means = [float(np.mean(dist_vals[d])) for d in dists]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))

axes[0].bar([str(d) for d in dists], means, color="#6366f1", width=0.6)
for i, v in enumerate(means):
    axes[0].text(i, v + 0.004, f"{v:.3f}", ha="center", fontsize=8, fontweight="bold")
axes[0].set_title("(a) |Tương quan| trung bình theo khoảng cách cột",
                  fontweight="bold")
axes[0].set_xlabel("Khoảng cách giữa hai cột trong file CSV")
axes[0].set_ylabel("|Pearson| trung bình")
axes[0].set_ylim(0, max(means) * 1.25)

# Một mẫu bệnh nhân vẽ như "tín hiệu 1D" — cái mà Conv1D sẽ nhìn thấy
sample = df.loc[3, feat_cols].values.astype(float)
axes[1].plot(range(1, n_f + 1), sample, "o-", color="#ef4444", linewidth=2, markersize=7)
axes[1].set_xticks(range(1, n_f + 1))
axes[1].set_xticklabels(feat_cols, rotation=38, ha="right", fontsize=8)
axes[1].set_title('(b) Một bệnh nhân nhìn như "chuỗi 1D" dưới mắt Conv1D',
                  fontweight="bold")
axes[1].set_ylabel("Giá trị thô")

plt.tight_layout()
plt.savefig(FIG / "c1_fig2_locality.png", bbox_inches="tight")
plt.show()

n_pairs = {d: len(v) for d, v in dist_vals.items()}
print("Tương quan trung bình theo khoảng cách cột:")
for d, m in zip(dists, means):
    print(f"  cách {d} cột: {m:.4f}   (trung bình trên {n_pairs[d]} cặp)")

near = np.mean([m for d, m in zip(dists, means) if d <= 3])
far = np.mean([m for d, m in zip(dists, means) if d >= 4])
i_max = int(np.argmax(corr_abs - np.eye(n_f)))
r_, c_ = divmod(i_max, n_f)
print(f"\nGần (cách 1-3 cột) : {near:.4f}")
print(f"Xa  (cách 4-7 cột) : {far:.4f}")
print("\nNhận xét:")
print("  • Không có xu hướng 'càng gần càng liên quan': các mức cách 1 đến 6 cột")
print(f"    đều dao động quanh 0,11-0,19 mà không giảm dần theo khoảng cách.")
print(f"  • Điểm 0,54 ở mức 'cách 7 cột' chỉ dựa trên ĐÚNG MỘT cặp:")
print(f"    {FEATURES_PREVIEW[r_]} ↔ {FEATURES_PREVIEW[c_]} (|r| = {corr_abs[r_, c_]:.3f}).")
print("    Đáng chú ý là cặp tương quan mạnh nhất toàn bảng lại nằm ở HAI ĐẦU")
print("    XA NHAU NHẤT của chuỗi — điều ngược hẳn với giả định cục bộ, và là")
print("    cặp mà kernel dài 3 KHÔNG BAO GIỜ nhìn thấy cùng lúc.")
print("\n  => Conv1D trên dữ liệu bảng là một thí nghiệm sư phạm, không phải")
print("     khẳng định CNN là mô hình tối ưu cho loại dữ liệu này (slide 8).")

Tương quan trung bình theo khoảng cách cột:
  cách 1 cột: 0.1855   (trung bình trên 7 cặp)
  cách 2 cột: 0.1502   (trung bình trên 6 cặp)
  cách 3 cột: 0.1842   (trung bình trên 5 cặp)
  cách 4 cột: 0.1125   (trung bình trên 4 cặp)
  cách 5 cột: 0.1315   (trung bình trên 3 cặp)
  cách 6 cột: 0.1485   (trung bình trên 2 cặp)
  cách 7 cột: 0.5443   (trung bình trên 1 cặp)

Gần (cách 1-3 cột) : 0.1733
Xa  (cách 4-7 cột) : 0.2342

Nhận xét:
  • Không có xu hướng 'càng gần càng liên quan': các mức cách 1 đến 6 cột
    đều dao động quanh 0,11-0,19 mà không giảm dần theo khoảng cách.
  • Điểm 0,54 ở mức 'cách 7 cột' chỉ dựa trên ĐÚNG MỘT cặp:
    Pregnancies ↔ Age (|r| = 0.544).
    Đáng chú ý là cặp tương quan mạnh nhất toàn bảng lại nằm ở HAI ĐẦU
    XA NHAU NHẤT của chuỗi — điều ngược hẳn với giả định cục bộ, và là
    cặp mà kernel dài 3 KHÔNG BAO GIỜ nhìn thấy cùng lúc.

  => Conv1D trên dữ liệu bảng là một thí nghiệm sư phạm, không phải
     khẳng định CNN là mô hình tối ưu cho loại dữ 

C:\Users\admin\AppData\Local\Temp\ipykernel_23616\3681519697.py:41: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2. Tiền xử lý — chống rò rỉ dữ liệu (Data Leakage)

Quy tắc bắt buộc: **mọi thống kê (median, mean, std) chỉ được ước lượng trên
tập huấn luyện**, rồi áp dụng nguyên si sang tập validation và test. Nếu tính
median trên toàn bộ dữ liệu thì thông tin của tập test đã rò rỉ vào quá trình
huấn luyện và điểm đánh giá sẽ bị thổi phồng.

In [7]:
# ============================================================================
# KHỐI 6 — Tiền xử lý có kiểm soát rò rỉ dữ liệu
# Thứ tự các bước:
#   1. đổi 0 thành NaN ở các cột sinh lý để đánh dấu đúng bản chất thiếu dữ liệu;
#   2. chia 70% train / 15% validation / 15% test, stratify theo nhãn;
#   3. TÍNH trung vị CHỈ trên tập train rồi dùng nó điền cho cả val và test;
#   4. StandardScaler cũng chỉ fit trên train.
# ============================================================================
X_raw = df.drop(columns=["Outcome"]).copy()
y = df["Outcome"].values.astype(float).reshape(-1, 1)
FEATURES = list(X_raw.columns)

for c in ZERO_AS_NAN:
    X_raw[c] = X_raw[c].replace(0, np.nan)

X_tmp, X_test, y_tmp, y_test = train_test_split(
    X_raw, y, test_size=0.15, random_state=SEED, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(
    X_tmp, y_tmp, test_size=0.1765, random_state=SEED, stratify=y_tmp)

medians = X_train.median()
X_train = X_train.fillna(medians)
X_val = X_val.fillna(medians)
X_test = X_test.fillna(medians)

scaler = StandardScaler().fit(X_train)
Xtr_flat = scaler.transform(X_train)
Xva_flat = scaler.transform(X_val)
Xte_flat = scaler.transform(X_test)

print(f"Train: {Xtr_flat.shape}  |  Val: {Xva_flat.shape}  |  Test: {Xte_flat.shape}")
print(f"Tỉ lệ dương tính  train={y_train.mean():.3f}  val={y_val.mean():.3f}  test={y_test.mean():.3f}")
print("\nMedian ước lượng trên TRAIN:")
print(medians.round(2).to_string())

Train: (536, 8)  |  Val: (116, 8)  |  Test: (116, 8)
Tỉ lệ dương tính  train=0.349  val=0.353  test=0.345

Median ước lượng trên TRAIN:
Pregnancies                   3.00
Glucose                     118.00
BloodPressure                72.00
SkinThickness                28.00
Insulin                     125.50
BMI                          32.00
DiabetesPedigreeFunction      0.38
Age                          29.00


### 2.1. Đổi hình dạng tensor: từ bảng sang chuỗi 1D

MLP nhận `(N, 8)`. CNN 1D nhận `(N, C_in, L)` — ta coi mỗi bệnh nhân là một
**chuỗi dài 8 với 1 kênh**, tức `(N, 1, 8)`. Đây là toàn bộ "phép biến hình"
cần thiết để chuyển bài toán bảng sang bài toán tích chập.

In [8]:
# ============================================================================
# KHỐI 7 — Thêm trục kênh: (N, 8) -> (N, 1, 8)
# Quy ước channels-first giống PyTorch. Bản TensorFlow dùng channels-last nên
# lúc đó ta sẽ transpose sang (N, 8, 1).
# ============================================================================
Xtr = Xtr_flat[:, None, :]
Xva = Xva_flat[:, None, :]
Xte = Xte_flat[:, None, :]

print("Sau khi thêm trục kênh (channels-first, dùng cho NumPy và PyTorch):")
print(f"  Xtr: {Xtr.shape}   (N, C_in, L)")
print(f"  Xva: {Xva.shape}")
print(f"  Xte: {Xte.shape}")
print("\nMột mẫu cụ thể — chuỗi 8 giá trị đã chuẩn hoá mà kernel sẽ trượt qua:")
print(np.round(Xtr[0, 0], 3))

Sau khi thêm trục kênh (channels-first, dùng cho NumPy và PyTorch):
  Xtr: (536, 1, 8)   (N, C_in, L)
  Xva: (116, 1, 8)
  Xte: (116, 1, 8)

Một mẫu cụ thể — chuỗi 8 giá trị đã chuẩn hoá mà kernel sẽ trượt qua:
[-1.133  0.253  3.095  1.919 -0.104  5.221 -0.462 -0.638]


## 3. Bản chất toán học của tích chập 1D — ví dụ tính bằng tay

Với đầu vào $x$ và kernel $k$ kích thước $m$, tích chập "valid" bước nhảy 1 là:

$$z_i = \sum_{j=0}^{m-1} k_j \, x_{i+j} + b$$

Lấy đúng ví dụ của slide: $x = [2, 1, 3, 4, 2]$, $k = [0.5, -1, 0.5]$, $b = 0$.

| Vị trí | Cửa sổ | Phép tính | Kết quả |
|---|---|---|---|
| $z_1$ | $[2, 1, 3]$ | $2(0.5) + 1(-1) + 3(0.5) = 1.0 - 1.0 + 1.5$ | $\mathbf{1.5}$ |
| $z_2$ | $[1, 3, 4]$ | $1(0.5) + 3(-1) + 4(0.5) = 0.5 - 3.0 + 2.0$ | $-0.5$ |
| $z_3$ | $[3, 4, 2]$ | $3(0.5) + 4(-1) + 2(0.5) = 1.5 - 4.0 + 1.0$ | $-1.5$ |

Sau đó $h = \mathrm{ReLU}(z) = [1.5,\; 0,\; 0]$.

> **Ghi chú về slide.** Slide 12 và 18 in $z_1 = 1$. Cộng lại từng số hạng thì
> $1.0 - 1.0 + 1.5 = 1.5$, nên giá trị đúng là $z_1 = 1.5$; $z_2$ và $z_3$ trên
> slide 19 thì chính xác. Báo cáo dùng giá trị đã kiểm chứng bằng cả tính tay
> lẫn ba nền tảng NumPy/PyTorch/TensorFlow (xem khối kiểm chứng ngay dưới).

In [9]:
import numpy as np

# ============================================================================
# 1. HÀM KÍCH HOẠT
# ============================================================================


def relu(z):
    """f(z) = max(0, z)."""
    return np.maximum(0.0, z)


def relu_grad(z):
    """f'(z) = 1 nếu z > 0, ngược lại 0. Tại z = 0 ta quy ước đạo hàm bằng 0."""
    return (z > 0).astype(z.dtype)


def sigmoid(z):
    """Ổn định số học: tách nhánh z >= 0 và z < 0 để exp() không tràn số."""
    out = np.empty_like(z, dtype=float)
    pos = z >= 0
    out[pos] = 1.0 / (1.0 + np.exp(-z[pos]))
    ez = np.exp(z[~pos])
    out[~pos] = ez / (1.0 + ez)
    return out


def softmax(z):
    """Trừ max theo hàng trước khi exp — kỹ thuật log-sum-exp chống tràn số."""
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)


# ============================================================================
# 2. TÍCH CHẬP 1D Ở MỨC HÀM
# ============================================================================
#
# Định nghĩa toán học dùng trong báo cáo (tích chập 'valid', stride 1):
#
#     z[n, o, l] = sum_{c=0}^{C_in-1} sum_{k=0}^{K-1} W[o, c, k] * x[n, c, l+k] + b[o]
#
# Bản `conv1d_naive` viết đúng ba vòng lặp như công thức trên — dễ đọc, dùng để
# kiểm chứng. Bản `conv1d_forward` dùng thủ thuật im2col nên nhanh hơn hàng chục
# lần mà vẫn cho kết quả giống hệt (notebook có assert đối chiếu hai bản).


def conv1d_naive(X, W, b):
    """Tích chập theo đúng định nghĩa toán học — ba vòng lặp tường minh.

    Chậm, nhưng là bản "nguyên văn công thức" để đối chiếu với bản im2col.
    """
    N, C_in, L_in = X.shape
    C_out, C_in_w, K = W.shape
    assert C_in == C_in_w, f"Số kênh vào không khớp: X có {C_in}, W cần {C_in_w}"
    L_out = L_in - K + 1

    Z = np.zeros((N, C_out, L_out))
    for o in range(C_out):          # từng bộ lọc
        for l in range(L_out):      # từng vị trí cửa sổ trượt
            for k in range(K):      # từng ô trong kernel
                Z[:, o, l] += np.sum(W[o, :, k] * X[:, :, l + k], axis=1)
            Z[:, o, l] += b[o]
    return Z


def im2col_1d(X, K):
    """Dàn mọi cửa sổ trượt thành một ma trận để biến tích chập thành nhân ma trận.

    Trả về `cols` có shape (N, C_in * K, L_out) với quy ước chỉ số:

        cols[n, c * K + k, l] = X[n, c, l + k]

    Nhờ layout `c * K + k`, mảng W shape (C_out, C_in, K) chỉ cần `reshape`
    (không cần transpose) là khớp: W.reshape(C_out, C_in*K)[o, c*K+k] = W[o,c,k].
    """
    N, C_in, L_in = X.shape
    L_out = L_in - K + 1
    cols = np.empty((N, C_in * K, L_out), dtype=X.dtype)
    for k in range(K):
        # Bước nhảy K: chỉ số k, K+k, 2K+k, ... đúng là c*K+k với c = 0, 1, 2, ...
        cols[:, k::K, :] = X[:, :, k:k + L_out]
    return cols


def col2im_1d(dcols, x_shape, K):
    """Phép chuyển vị của im2col: cộng dồn gradient về đúng vị trí trong X.

    Một phần tử x[n, c, i] tham gia nhiều cửa sổ khác nhau, nên gradient của nó
    là TỔNG các đóng góp — đây chính là chỗ phép cộng dồn (`+=`) là bắt buộc.
    """
    N, C_in, L_in = x_shape
    L_out = L_in - K + 1
    dX = np.zeros(x_shape, dtype=float)
    for k in range(K):
        dX[:, :, k:k + L_out] += dcols[:, k::K, :]
    return dX


def conv1d_forward(X, W, b):
    """Tích chập 'valid' stride 1 bằng im2col. Trả về (Z, cache)."""
    C_out, C_in, K = W.shape
    cols = im2col_1d(X, K)                              # (N, C_in*K, L_out)
    Wr = W.reshape(C_out, C_in * K)                     # (C_out, C_in*K)
    Z = np.einsum("ok,nkl->nol", Wr, cols, optimize=True) + b[None, :, None]
    return Z, (cols, X.shape, W.shape)


def conv1d_backward(dZ, W, cache):
    """Lan truyền ngược qua tích chập. Trả về (dX, dW, db).

    Cho dZ = dL/dZ shape (N, C_out, L_out), ba công thức suy ra trực tiếp từ
    z = W·cols + b:

        dW[o, c, k] = sum_{n,l} dZ[n,o,l] * cols[n, c*K+k, l]
        db[o]       = sum_{n,l} dZ[n,o,l]
        dcols       = W^T · dZ   ->  dX = col2im(dcols)

    Chú ý db là tổng trên CẢ batch VÀ mọi vị trí l: một bias duy nhất được dùng
    lại ở mọi vị trí (chia sẻ trọng số), nên gradient của nó gom hết đóng góp.
    Điều tương tự xảy ra với dW — đây chính là dấu vết toán học của việc chia sẻ
    trọng số, và là điểm khác biệt cốt lõi so với tầng Dense.
    """
    cols, x_shape, w_shape = cache
    C_out, C_in, K = w_shape

    dW = np.einsum("nol,nkl->ok", dZ, cols, optimize=True).reshape(w_shape)
    db = dZ.sum(axis=(0, 2))

    Wr = W.reshape(C_out, C_in * K)
    dcols = np.einsum("ok,nol->nkl", Wr, dZ, optimize=True)
    dX = col2im_1d(dcols, x_shape, K)
    return dX, dW, db


# ============================================================================
# 3. CÁC LỚP TẦNG
# ============================================================================


class Layer:
    """Giao diện chung. `params()` trả về list [(tên, mảng_trọng_số, hàm_gán)]."""

    trainable = False

    def forward(self, X, training=False):
        raise NotImplementedError

    def backward(self, dOut):
        raise NotImplementedError

    def param_list(self):
        """Trả về list các mảng tham số (để Adam cập nhật tại chỗ)."""
        return []

    def grad_list(self):
        return []

    def describe(self, in_shape):
        """(mô tả, out_shape, số tham số) — dùng để in bảng đặc tả tensor."""
        return (type(self).__name__, in_shape, 0)

    def to_dict(self, decimals):
        return {"type": type(self).__name__.lower()}


class Conv1D(Layer):
    """Tầng tích chập 1D: (N, C_in, L) -> (N, C_out, L-K+1).

    Khởi tạo He (Kaiming) với fan_in = C_in * K, phù hợp với ReLU đứng sau.
    """

    trainable = True

    def __init__(self, c_in, c_out, k, rng):
        self.c_in, self.c_out, self.k = c_in, c_out, k
        fan_in = c_in * k
        self.W = rng.normal(0.0, np.sqrt(2.0 / fan_in), (c_out, c_in, k))
        self.b = np.zeros(c_out)
        self.dW = np.zeros_like(self.W)
        self.db = np.zeros_like(self.b)

    def forward(self, X, training=False):
        Z, self._cache = conv1d_forward(X, self.W, self.b)
        return Z

    def backward(self, dZ):
        dX, self.dW, self.db = conv1d_backward(dZ, self.W, self._cache)
        return dX

    def param_list(self):
        return [self.W, self.b]

    def grad_list(self):
        return [self.dW, self.db]

    def describe(self, in_shape):
        C, L = in_shape
        out = (self.c_out, L - self.k + 1)
        n = self.c_out * self.c_in * self.k + self.c_out
        return (f"Conv1D(C {self.c_in}→{self.c_out}, K={self.k})", out, n)

    def to_dict(self, decimals):
        return {
            "type": "conv1d",
            "c_in": self.c_in, "c_out": self.c_out, "k": self.k,
            "W": np.round(self.W, decimals).tolist(),
            "b": np.round(self.b, decimals).tolist(),
        }


class ReLU(Layer):
    def forward(self, X, training=False):
        self._z = X
        return relu(X)

    def backward(self, dOut):
        return dOut * relu_grad(self._z)

    def describe(self, in_shape):
        return ("ReLU", in_shape, 0)

    def to_dict(self, decimals):
        return {"type": "relu"}


class MaxPool1D(Layer):
    """Gom cụm cực đại theo cửa sổ không chồng lấn, size = stride = `p`.

    Nếu L không chia hết cho p, phần dư ở cuối bị bỏ (floor) — giống hành vi
    mặc định của nn.MaxPool1d và MaxPooling1D.
    """

    def __init__(self, p=2):
        self.p = p

    def forward(self, X, training=False):
        N, C, L = X.shape
        p = self.p
        L_out = L // p
        Xc = X[:, :, :L_out * p].reshape(N, C, L_out, p)
        self._argmax = Xc.argmax(axis=3)
        self._shape = X.shape
        return Xc.max(axis=3)

    def backward(self, dOut):
        N, C, L = self._shape
        p = self.p
        L_out = L // p
        dX = np.zeros((N, C, L_out * p))
        # Chỉ ô thắng (đạt cực đại) nhận gradient, các ô còn lại nhận 0.
        n_i, c_i, l_i = np.ogrid[:N, :C, :L_out]
        dXc = dX.reshape(N, C, L_out, p)
        dXc[n_i, c_i, l_i, self._argmax] = dOut
        full = np.zeros(self._shape)
        full[:, :, :L_out * p] = dXc.reshape(N, C, L_out * p)
        return full

    def describe(self, in_shape):
        C, L = in_shape
        return (f"MaxPool1D(p={self.p})", (C, L // self.p), 0)

    def to_dict(self, decimals):
        return {"type": "maxpool1d", "p": self.p}


class GlobalMaxPool1D(Layer):
    """Lấy giá trị lớn nhất trên toàn trục thời gian: (N, C, L) -> (N, C).

    Dùng cho văn bản: mỗi bộ lọc trả lời "mẫu cục bộ mà tôi phụ trách có xuất
    hiện ở đâu đó trong câu hay không", nên độ dài câu không còn ảnh hưởng tới
    số chiều đầu ra — đó là cách CNN xử lý câu dài ngắn khác nhau.
    """

    def forward(self, X, training=False):
        self._argmax = X.argmax(axis=2)
        self._shape = X.shape
        return X.max(axis=2)

    def backward(self, dOut):
        N, C, L = self._shape
        dX = np.zeros(self._shape)
        n_i, c_i = np.ogrid[:N, :C]
        dX[n_i, c_i, self._argmax] = dOut
        return dX

    def describe(self, in_shape):
        C, L = in_shape
        return ("GlobalMaxPool1D", (C,), 0)

    def to_dict(self, decimals):
        return {"type": "globalmaxpool1d"}


class Flatten(Layer):
    """(N, C, L) -> (N, C*L). Thứ tự dàn phẳng là C trước, L sau (C-order)."""

    def forward(self, X, training=False):
        self._shape = X.shape
        return X.reshape(X.shape[0], -1)

    def backward(self, dOut):
        return dOut.reshape(self._shape)

    def describe(self, in_shape):
        return ("Flatten", (int(np.prod(in_shape)),), 0)

    def to_dict(self, decimals):
        return {"type": "flatten"}


class Dense(Layer):
    """Tầng kết nối đầy đủ: (N, D) -> (N, M), W shape (D, M)."""

    trainable = True

    def __init__(self, d_in, d_out, rng):
        self.d_in, self.d_out = d_in, d_out
        self.W = rng.normal(0.0, np.sqrt(2.0 / d_in), (d_in, d_out))
        self.b = np.zeros(d_out)
        self.dW = np.zeros_like(self.W)
        self.db = np.zeros_like(self.b)

    def forward(self, X, training=False):
        self._x = X
        return X @ self.W + self.b

    def backward(self, dOut):
        self.dW = self._x.T @ dOut
        self.db = dOut.sum(axis=0)
        return dOut @ self.W.T

    def param_list(self):
        return [self.W, self.b]

    def grad_list(self):
        return [self.dW, self.db]

    def describe(self, in_shape):
        n = self.d_in * self.d_out + self.d_out
        return (f"Dense({self.d_in}→{self.d_out})", (self.d_out,), n)

    def to_dict(self, decimals):
        return {
            "type": "dense",
            "d_in": self.d_in, "d_out": self.d_out,
            "W": np.round(self.W, decimals).tolist(),
            "b": np.round(self.b, decimals).tolist(),
        }


class Dropout(Layer):
    """Inverted dropout — chia cho keep_prob ngay lúc train nên lúc suy luận
    không phải chỉnh gì, trọng số xuất ra JSON dùng trực tiếp được."""

    def __init__(self, rate, rng):
        self.rate = rate
        self.rng = rng

    def forward(self, X, training=False):
        if not training or self.rate <= 0.0:
            self._mask = None
            return X
        keep = 1.0 - self.rate
        self._mask = (self.rng.random(X.shape) < keep) / keep
        return X * self._mask

    def backward(self, dOut):
        return dOut if self._mask is None else dOut * self._mask

    def describe(self, in_shape):
        return (f"Dropout(p={self.rate})", in_shape, 0)

    def to_dict(self, decimals):
        return {"type": "dropout", "rate": self.rate}


class Embedding(Layer):
    """Tra bảng nhúng cho văn bản: (N, L) chỉ số nguyên -> (N, C_emb, L).

    Đầu ra đã ở dạng channels-first để Conv1D dùng trực tiếp. Chỉ số 0 được
    dành riêng cho token đệm (PAD) và vector của nó bị ghim bằng 0 (cả trong
    khởi tạo lẫn sau mỗi bước cập nhật), nên phần đệm không đóng góp gì vào
    tích chập.
    """

    trainable = True

    def __init__(self, vocab_size, dim, rng, pad_idx=0):
        self.vocab_size, self.dim, self.pad_idx = vocab_size, dim, pad_idx
        self.E = rng.normal(0.0, 0.05, (vocab_size, dim))
        self.E[pad_idx] = 0.0
        self.dE = np.zeros_like(self.E)

    def forward(self, idx, training=False):
        self._idx = idx.astype(np.int64)
        # (N, L, dim) -> (N, dim, L)
        return self.E[self._idx].transpose(0, 2, 1)

    def backward(self, dOut):
        # dOut: (N, dim, L) -> (N, L, dim), rồi cộng dồn theo chỉ số token.
        g = dOut.transpose(0, 2, 1)
        self.dE = np.zeros_like(self.E)
        # np.add.at cộng dồn đúng khi một token xuất hiện nhiều lần trong batch
        # (phép gán thường sẽ ghi đè và làm mất gradient).
        np.add.at(self.dE, self._idx.ravel(), g.reshape(-1, self.dim))
        self.dE[self.pad_idx] = 0.0
        return None  # Embedding là tầng đầu tiên — không cần truyền ngược nữa

    def param_list(self):
        return [self.E]

    def grad_list(self):
        return [self.dE]

    def describe(self, in_shape):
        (L,) = in_shape
        return (f"Embedding({self.vocab_size}→{self.dim})", (self.dim, L),
                self.vocab_size * self.dim)

    def to_dict(self, decimals):
        return {
            "type": "embedding",
            "vocab_size": self.vocab_size, "dim": self.dim, "pad_idx": self.pad_idx,
            "E": np.round(self.E, decimals).tolist(),
        }


# ============================================================================
# 4. BỘ CHỨA TUẦN TỰ + HUẤN LUYỆN
# ============================================================================


class CNN1D:
    """CNN 1D tuần tự, tự cài forward/backward/Adam.

    `task` quyết định đầu ra và hàm mất mát:
        binary      : 1 nơ-ron + Sigmoid  + Binary Cross-Entropy
        regression  : 1 nơ-ron + Linear   + Mean Squared Error
        multiclass  : C nơ-ron + Softmax  + Categorical Cross-Entropy

    Với cả ba, đạo hàm của loss theo pre-activation cuối rút gọn về (ŷ − y) —
    đó chính là lý do ta ghép Sigmoid/Softmax với Cross-Entropy và Linear
    với MSE, chi tiết ở Chương I của báo cáo.
    """

    def __init__(self, layers, task="binary", lr=1e-3, l2=0.0, seed=42,
                 class_weight=None, clip_norm=None):
        assert task in {"binary", "regression", "multiclass"}
        self.layers = layers
        self.task = task
        self.lr = lr
        self.l2 = l2
        self.clip_norm = clip_norm
        self.class_weight = None if class_weight is None else np.asarray(class_weight, float)
        self.rng = np.random.default_rng(seed)

        # Trạng thái Adam: một cặp (m, v) cho mỗi mảng tham số.
        self._params = [p for L in layers for p in L.param_list()]
        self.m = [np.zeros_like(p) for p in self._params]
        self.v = [np.zeros_like(p) for p in self._params]
        self.t = 0
        self.history = {"train_loss": [], "val_loss": [], "train_metric": [], "val_metric": []}

    # ----------------------------- forward ---------------------------------
    def forward(self, X, training=False):
        A = X
        for L in self.layers:
            A = L.forward(A, training=training)
        Z = A                                  # pre-activation của head
        if self.task == "binary":
            return sigmoid(Z), Z
        if self.task == "multiclass":
            return softmax(Z), Z
        return Z, Z

    # ------------------------------ loss -----------------------------------
    def _sample_w(self, y_true):
        if self.class_weight is None or self.task != "multiclass":
            return None
        return self.class_weight[y_true.argmax(1)].reshape(-1, 1)

    def loss(self, y_pred, y_true, logits=None):
        """Hàm mất mát. Nếu truyền `logits` (pre-activation của head) thì dùng dạng
        log-sum-exp / log-sigmoid.

        Vì sao phải quan tâm: cách viết "ngây thơ" `-log(p + eps)` gài một sai số
        hệ thống bằng eps/p. Khi mạng gán cho lớp đúng một xác suất rất nhỏ
        (p ~ 1e-8, hay gặp ở epoch đầu), sai số đó lên tới ~1e-4 tương đối và
        KHÔNG khớp với gradient giải tích (ŷ − y) — đủ để phép kiểm tra gradient
        bằng sai phân số thất bại dù backward hoàn toàn đúng.

        Dạng logit dưới đây không cần eps, ổn định số học, và nhất quán tuyệt đối
        với dZ = ŷ − y. Đây cũng chính là lý do PyTorch khuyên dùng
        `binary_cross_entropy_with_logits` thay cho `sigmoid` rồi `log`.
        """
        n = y_true.shape[0]
        eps = 1e-12
        if self.task == "binary":
            if logits is not None:
                z = logits
                # max(z,0) − z·y + log(1 + exp(−|z|)) — BCE-with-logits ổn định.
                base = float(np.mean(np.maximum(z, 0.0) - z * y_true
                                     + np.log1p(np.exp(-np.abs(z)))))
            else:
                base = -np.mean(y_true * np.log(y_pred + eps)
                                + (1 - y_true) * np.log(1 - y_pred + eps))
        elif self.task == "multiclass":
            if logits is not None:
                z = logits - logits.max(axis=1, keepdims=True)
                log_p = z - np.log(np.exp(z).sum(axis=1, keepdims=True))
                per = -np.sum(y_true * log_p, axis=1, keepdims=True)
            else:
                per = -np.sum(y_true * np.log(y_pred + eps), axis=1, keepdims=True)
            w = self._sample_w(y_true)
            base = float(np.sum(per if w is None else per * w) / n)
        else:
            base = float(np.mean((y_pred - y_true) ** 2))
        reg = self.l2 * sum(np.sum(p * p) for p in self._params) / (2 * n) if self.l2 else 0.0
        return base + reg

    # ---------------------------- backward ---------------------------------
    def backward(self, y_pred, y_true):
        n = y_true.shape[0]
        dZ = (y_pred - y_true) / n
        if self.task == "regression":
            dZ = 2.0 * dZ
        w = self._sample_w(y_true)
        if w is not None:
            dZ = dZ * w

        d = dZ
        for L in reversed(self.layers):
            d = L.backward(d)
            if d is None:       # đã tới tầng Embedding
                break

        if self.l2:
            for L in self.layers:
                if L.trainable:
                    for p, g in zip(L.param_list(), L.grad_list()):
                        g += self.l2 * p / n

    # ------------------------------ Adam -----------------------------------
    def _adam(self, beta1=0.9, beta2=0.999, eps=1e-8):
        grads = [g for L in self.layers for g in L.grad_list()]

        if self.clip_norm:
            # Cắt chuẩn gradient toàn cục: giữ hướng, chỉ co độ dài. Bài văn bản
            # có gradient nhảy vọt khi một n-gram hiếm xuất hiện, nên cần thứ này.
            total = np.sqrt(sum(float(np.sum(g * g)) for g in grads))
            if total > self.clip_norm:
                scale = self.clip_norm / (total + 1e-12)
                grads = [g * scale for g in grads]

        self.t += 1
        for i, (p, g) in enumerate(zip(self._params, grads)):
            self.m[i] = beta1 * self.m[i] + (1 - beta1) * g
            self.v[i] = beta2 * self.v[i] + (1 - beta2) * (g * g)
            mhat = self.m[i] / (1 - beta1 ** self.t)
            vhat = self.v[i] / (1 - beta2 ** self.t)
            p -= self.lr * mhat / (np.sqrt(vhat) + eps)   # cập nhật TẠI CHỖ

        # Giữ vector PAD bằng 0 sau mỗi bước (Adam có thể đẩy nó lệch khỏi 0).
        for L in self.layers:
            if isinstance(L, Embedding):
                L.E[L.pad_idx] = 0.0

    # ------------------------------ metric ---------------------------------
    def _metric(self, X, y, batch=512):
        p = self.predict_proba(X, batch=batch)
        if self.task == "binary":
            return float(np.mean((p >= 0.5).astype(int) == y))
        if self.task == "multiclass":
            return float(np.mean(p.argmax(1) == y.argmax(1)))
        ss_res = float(np.sum((y - p) ** 2))
        ss_tot = float(np.sum((y - y.mean()) ** 2))
        return 1.0 - ss_res / ss_tot            # R^2

    # ------------------------------- fit -----------------------------------
    def fit(self, X, y, X_val=None, y_val=None, epochs=100, batch_size=32,
            verbose_every=10, patience=None, eval_batch=512, eval_subset=None,
            val_score_fn=None):
        """Chu trình 4 bước mỗi mini-batch: Forward → Loss → Backward → Update.

        `eval_subset`: nếu đặt, loss/metric TRAIN mỗi epoch chỉ tính trên một mẫu
        con cố định cỡ này (chọn một lần, không đổi giữa các epoch) thay vì toàn
        bộ tập train. Chỉ dùng cho bài văn bản 23k mẫu, nơi một lượt forward đầy
        đủ mỗi epoch đắt hơn cả việc huấn luyện. Early stopping vẫn dựa trên
        validation đầy đủ nên không ảnh hưởng tới việc chọn mô hình.

        `val_score_fn(y_true_onehot, y_prob) -> float`: nếu đặt, early stopping
        và việc giữ trọng số tốt nhất sẽ **cực đại hoá** điểm này thay vì cực
        tiểu hoá val_loss.

        Vì sao cần: với dữ liệu mất cân bằng nặng và cross-entropy CÓ TRỌNG SỐ
        LỚP, val_loss dao động mạnh và đạt cực tiểu rất sớm, trong khi Macro-F1
        vẫn còn đang lên. Chọn mô hình theo val_loss khi đó dừng quá sớm và cho
        mô hình kém hẳn. Tiêu chí chọn mô hình phải là chỉ số ta thật sự quan tâm.
        """
        n = X.shape[0]
        # Quy ước nội bộ: luôn CỰC TIỂU `best_val`. Khi dùng val_score_fn ta lấy
        # dấu âm của điểm số, nên một nhánh early-stopping duy nhất phục vụ cả hai.
        best_val, best_state, wait = np.inf, None, 0
        self.history.setdefault("val_score", [])

        if eval_subset is not None and eval_subset < n:
            sub = self.rng.choice(n, size=eval_subset, replace=False)
            X_tr_eval, y_tr_eval = X[sub], y[sub]
        else:
            X_tr_eval, y_tr_eval = X, y

        for ep in range(1, epochs + 1):
            idx = self.rng.permutation(n)
            for s in range(0, n, batch_size):
                sl = idx[s:s + batch_size]
                xb, yb = X[sl], y[sl]
                yp, _ = self.forward(xb, training=True)     # (1) Forward
                self.backward(yp, yb)                      # (3) Backward
                self._adam()                               # (4) Update

            tr_pred, tr_logit = self._eval(X_tr_eval, batch=eval_batch)
            tr_loss = self.loss(tr_pred, y_tr_eval, logits=tr_logit)   # (2) Loss
            self.history["train_loss"].append(tr_loss)
            self.history["train_metric"].append(self._metric(X_tr_eval, y_tr_eval, eval_batch))

            if X_val is not None:
                va_pred, va_logit = self._eval(X_val, batch=eval_batch)
                va_loss = self.loss(va_pred, y_val, logits=va_logit)
                self.history["val_loss"].append(va_loss)
                self.history["val_metric"].append(self._metric(X_val, y_val, eval_batch))

                if val_score_fn is not None:
                    score = float(val_score_fn(y_val, va_pred))
                    self.history["val_score"].append(score)
                    watched, label = -score, "val_score"
                else:
                    watched, label = va_loss, "val_loss"

                if patience is not None:
                    if watched < best_val - 1e-6:
                        best_val, wait = watched, 0
                        best_state = [p.copy() for p in self._params]
                    else:
                        wait += 1
                        if wait >= patience:
                            if verbose_every:
                                shown = -best_val if val_score_fn is not None else best_val
                                print(f"  ⏹ Early stopping tại epoch {ep} "
                                      f"({label} tốt nhất = {shown:.4f})")
                            break

            if verbose_every and (ep % verbose_every == 0 or ep == 1):
                msg = (f"  epoch {ep:4d} | train_loss={tr_loss:.4f} "
                       f"| train_metric={self.history['train_metric'][-1]:.4f}")
                if X_val is not None:
                    msg += (f" | val_loss={self.history['val_loss'][-1]:.4f}"
                            f" | val_metric={self.history['val_metric'][-1]:.4f}")
                    if val_score_fn is not None:
                        msg += f" | val_score={self.history['val_score'][-1]:.4f}"
                print(msg)

        if best_state is not None:
            # Trả tham số về trạng thái tốt nhất trên validation, ghi TẠI CHỖ để
            # self._params (và các layer) vẫn trỏ tới cùng mảng.
            for p, bp in zip(self._params, best_state):
                p[...] = bp
        return self

    # ---------------------------- inference --------------------------------
    def _eval(self, X, batch=512):
        """Suy luận theo lô, trả về (xác suất, logits). Logits cần cho `loss()`."""
        ps, zs = [], []
        for s in range(0, X.shape[0], batch):
            p, z = self.forward(X[s:s + batch], training=False)
            ps.append(p)
            zs.append(z)
        return np.concatenate(ps, axis=0), np.concatenate(zs, axis=0)

    def predict_proba(self, X, batch=512):
        return self._eval(X, batch)[0]

    def predict(self, X, threshold=0.5, batch=512):
        p = self.predict_proba(X, batch=batch)
        if self.task == "binary":
            return (p >= threshold).astype(int)
        if self.task == "multiclass":
            return p.argmax(1)
        return p

    # ------------------------- đặc tả và xuất JSON -------------------------
    def n_params(self):
        return int(sum(p.size for p in self._params))

    def shape_table(self, in_shape):
        """Bảng (tầng, đầu vào, đầu ra, số tham số) — phần đặc tả tensor của báo cáo."""
        rows, shape = [], tuple(in_shape)
        for L in self.layers:
            name, out, n = L.describe(shape)
            rows.append({"Tầng": name,
                         "Input shape": f"(batch, {', '.join(map(str, shape))})",
                         "Output shape": f"(batch, {', '.join(map(str, out))})",
                         "Số tham số": n,
                         "Huấn luyện được": "✓" if L.trainable else "—"})
            shape = out
        return rows

    def to_dict(self, decimals=6):
        head = {"binary": "sigmoid", "multiclass": "softmax", "regression": "linear"}[self.task]
        return {
            "kind": "cnn1d",
            "task": self.task,
            "head": head,
            "n_params": self.n_params(),
            "n_trainable_layers": sum(1 for L in self.layers if L.trainable),
            "layers": [L.to_dict(decimals) for L in self.layers],
        }


# ============================================================================
# 5. TIỆN ÍCH
# ============================================================================


def one_hot(y, n_classes):
    out = np.zeros((len(y), n_classes))
    out[np.arange(len(y)), y] = 1.0
    return out


def forward_reference(bundle, x_single):
    """Bản tham chiếu của thuật toán suy luận sẽ viết lại bằng JavaScript.

    Nhận đúng dict đã ghi ra `model_cnn.json` (nên mọi trọng số đã bị làm tròn)
    và MỘT mẫu. Dùng trong notebook để kiểm tra parity NumPy ↔ JSON ↔ JS: nếu
    hàm này khớp với mô hình gốc thì bản JS chỉ cần dịch nguyên văn là đúng.

    x_single:
        - bài bảng  : mảng (C_in, L) hoặc (L,) — sẽ được thêm trục kênh
        - bài văn bản: mảng (L,) chỉ số token nguyên
    """
    a = np.asarray(x_single)
    layers = bundle["layers"]

    if layers[0]["type"] == "embedding":
        a = a.astype(np.int64)[None, :]                    # (1, L)
    else:
        if a.ndim == 1:
            a = a[None, :]                                 # (C_in=1, L)
        a = a.astype(float)[None, ...]                     # (1, C_in, L)

    for layer in layers:
        t = layer["type"]
        if t == "embedding":
            E = np.asarray(layer["E"])
            a = E[a].transpose(0, 2, 1)
        elif t == "conv1d":
            W = np.asarray(layer["W"])
            b = np.asarray(layer["b"])
            a, _ = conv1d_forward(a, W, b)
        elif t == "relu":
            a = relu(a)
        elif t == "maxpool1d":
            p = layer["p"]
            N, C, L = a.shape
            L_out = L // p
            a = a[:, :, :L_out * p].reshape(N, C, L_out, p).max(axis=3)
        elif t == "globalmaxpool1d":
            a = a.max(axis=2)
        elif t == "flatten":
            a = a.reshape(a.shape[0], -1)
        elif t == "dense":
            a = a @ np.asarray(layer["W"]) + np.asarray(layer["b"])
        elif t == "dropout":
            pass                                           # suy luận: dropout vô hiệu
        else:
            raise ValueError(f"Tầng lạ trong bundle: {t}")

    head = bundle["head"]
    if head == "sigmoid":
        return float(sigmoid(a).ravel()[0])
    if head == "softmax":
        return softmax(a.reshape(1, -1))[0]
    return float(a.ravel()[0])

In [10]:
# ============================================================================
# KHỐI 9 — Kiểm chứng ví dụ số học bằng cả ba nền tảng
# Nếu ba nền tảng cùng cho [1.5, -0.5, -1.5] thì con số tính tay là đúng.
# ============================================================================
x_demo = np.array([[[2.0, 1.0, 3.0, 4.0, 2.0]]])     # (1, 1, 5)
k_demo = np.array([[[0.5, -1.0, 0.5]]])              # (1, 1, 3)
b_demo = np.array([0.0])

z_np, _ = conv1d_forward(x_demo, k_demo, b_demo)
z_naive = conv1d_naive(x_demo, k_demo, b_demo)

print("NumPy (im2col)        :", np.round(z_np.ravel(), 6))
print("NumPy (3 vòng lặp)    :", np.round(z_naive.ravel(), 6))
print("ReLU(z)               :", np.round(relu(z_np).ravel(), 6))
print()
print("Ví dụ max pooling của slide 14: [1, 5, 2, 4] với p = 2")
print("  ->", MaxPool1D(2).forward(np.array([[[1.0, 5.0, 2.0, 4.0]]])).ravel())
assert np.allclose(z_np.ravel(), [1.5, -0.5, -1.5]), "Ví dụ số học sai"
assert np.allclose(z_np, z_naive), "im2col không khớp bản naive"
print("\n✅ Bản im2col khớp tuyệt đối với bản viết theo đúng công thức toán học.")

NumPy (im2col)        : [ 1.5 -0.5 -1.5]
NumPy (3 vòng lặp)    : [ 1.5 -0.5 -1.5]
ReLU(z)               : [1.5 0.  0. ]

Ví dụ max pooling của slide 14: [1, 5, 2, 4] với p = 2
  -> [5. 4.]

✅ Bản im2col khớp tuyệt đối với bản viết theo đúng công thức toán học.


## 4. Kiến trúc CNN 5 tầng và định nghĩa 'Layer'

### Định nghĩa 'Layer' dùng trong báo cáo

Slide 16 yêu cầu nêu rõ cách đếm tầng. **Quy ước của báo cáo này:** chỉ các
phép biến đổi **có tham số huấn luyện được** mới được tính là một *layer* —
tức `Conv1D`, `Dense` và `Embedding`. Các phép `ReLU`, `MaxPool1D`, `Flatten`,
`Dropout` là **phép toán không tham số**, được xem là thành phần bên trong tầng
chứ không đếm riêng.

Theo quy ước đó, kiến trúc dưới đây có **5 tầng**, đúng như slide 16:

$$X \to \underbrace{\text{Conv}_1}_{1} \to \text{ReLU} \to \underbrace{\text{Conv}_2}_{2} \to \text{ReLU} \to \text{Pool} \to \underbrace{\text{Conv}_3}_{3} \to \text{ReLU} \to \text{Flatten} \to \underbrace{\text{Dense}_1}_{4} \to \text{ReLU} \to \underbrace{\text{Dense}_2}_{5} \to \sigma \to \hat{y}$$

Và toàn mạng vẫn chỉ là một **hợp thành hàm**, đúng tinh thần Lecture 03:

$$\hat{y} = f_5 \circ f_4 \circ f_3 \circ f_2 \circ f_1 (X)$$

In [11]:
# ============================================================================
# KHỐI 10 — Dựng CNN 5 tầng và in bảng đặc tả tensor
# Đường đi của độ dài chuỗi: 8 -> 6 -> 4 -> (pool) 2 -> 1
# HP là bộ siêu tham số đã dò trên tập VALIDATION (tiêu chí ROC-AUC), không
# phải trên tập test.
# ============================================================================
HP = dict(lr=2e-3, l2=1e-2, dropout=0.30, batch_size=32, epochs=400, patience=60)
print("Siêu tham số (chọn trên tập VALIDATION theo ROC-AUC):")
print(HP, "\n")


def build_cnn(seed=SEED, dropout=HP["dropout"], lr=HP["lr"], l2=HP["l2"], n_in=8):
    """CNN 5 tầng huấn luyện được cho bài phân loại nhị phân 8 đặc trưng."""
    r = np.random.default_rng(seed)
    layers = [
        Conv1D(1, 16, 3, r), ReLU(),        # tầng 1: (1, 8)  -> (16, 6)
        Conv1D(16, 16, 3, r), ReLU(),       # tầng 2: (16, 6) -> (16, 4)
        MaxPool1D(2),                       #         (16, 4) -> (16, 2)
        Conv1D(16, 32, 2, r), ReLU(),       # tầng 3: (16, 2) -> (32, 1)
        Flatten(),                          #         (32, 1) -> (32,)
        Dropout(dropout, r),
        Dense(32, 16, r), ReLU(),           # tầng 4: 32 -> 16
        Dense(16, 1, r),                    # tầng 5: 16 -> 1  (+ Sigmoid ở head)
    ]
    return CNN1D(layers=layers, task="binary", lr=lr, l2=l2, seed=seed)


cnn = build_cnn()
shape_rows = cnn.shape_table((1, 8))
shape_table = pd.DataFrame(shape_rows)
shape_table.loc[len(shape_table)] = ["TỔNG", "", "", shape_table["Số tham số"].sum(), ""]
print(f"Tổng tham số: {cnn.n_params():,} — "
      f"{cnn.to_dict()['n_trainable_layers']} tầng huấn luyện được")
shape_table

Siêu tham số (chọn trên tập VALIDATION theo ROC-AUC):
{'lr': 0.002, 'l2': 0.01, 'dropout': 0.3, 'batch_size': 32, 'epochs': 400, 'patience': 60} 

Tổng tham số: 2,449 — 5 tầng huấn luyện được


,Tầng,Input shape,Output shape,Số tham số,Huấn luyện được
0,"Conv1D(C 1→16, K=3)","(batch, 1, 8)","(batch, 16, 6)",64,✓
1,ReLU,"(batch, 16, 6)","(batch, 16, 6)",0,—
2,"Conv1D(C 16→16, K=3)","(batch, 16, 6)","(batch, 16, 4)",784,✓
3,ReLU,"(batch, 16, 4)","(batch, 16, 4)",0,—
4,MaxPool1D(p=2),"(batch, 16, 4)","(batch, 16, 2)",0,—
5,"Conv1D(C 16→32, K=2)","(batch, 16, 2)","(batch, 32, 1)",1056,✓
6,ReLU,"(batch, 32, 1)","(batch, 32, 1)",0,—
7,Flatten,"(batch, 32, 1)","(batch, 32)",0,—
8,Dropout(p=0.3),"(batch, 32)","(batch, 32)",0,—
9,Dense(32→16),"(batch, 32)","(batch, 16)",528,✓


### 4.1. So sánh không gian tham số: Dense vs Conv1D

Đây là chỗ thấy rõ ích lợi của **chia sẻ trọng số**. Một tầng Dense nối 8 đầu
vào với 16 nơ-ron cần $8 \times 16 + 16 = 144$ tham số. Tầng `Conv1D(1→16, K=3)`
chỉ cần $16 \times 1 \times 3 + 16 = 64$ tham số mà vẫn sinh ra 16 bản đồ đặc
trưng, vì **cùng một kernel được dùng lại ở mọi vị trí**.

In [12]:
# ============================================================================
# KHỐI 11 — Đếm tham số: Conv1D tiết kiệm bao nhiêu so với Dense tương đương
# ============================================================================
rows = []
for (c_in, c_out, k, L_in) in [(1, 16, 3, 8), (16, 16, 3, 6), (16, 32, 2, 2)]:
    L_out = L_in - k + 1
    conv_p = c_out * c_in * k + c_out
    dense_p = (c_in * L_in) * (c_out * L_out) + c_out * L_out
    rows.append({
        "Tầng": f"Conv1D({c_in}→{c_out}, K={k}) trên L={L_in}",
        "Tham số Conv1D": conv_p,
        "Dense tương đương": dense_p,
        "Tiết kiệm": f"{dense_p / conv_p:.1f}×",
    })
param_cmp = pd.DataFrame(rows)
print("Cùng một phép biến đổi hình dạng, Conv1D dùng ít tham số hơn nhiều:")
param_cmp

Cùng một phép biến đổi hình dạng, Conv1D dùng ít tham số hơn nhiều:


,Tầng,Tham số Conv1D,Dense tương đương,Tiết kiệm
0,"Conv1D(1→16, K=3) trên L=8",64,864,13.5×
1,"Conv1D(16→16, K=3) trên L=6",784,6208,7.9×
2,"Conv1D(16→32, K=2) trên L=2",1056,1056,1.0×


## 5. Kiểm chứng gradient viết tay (Gradient Checking)

Trước khi tin vào bất kỳ con số huấn luyện nào, phải chứng minh bản lan truyền
ngược tự viết là **đúng**. Ta so gradient giải tích với gradient sai phân số
trung tâm:

$$\frac{\partial L}{\partial \theta_i} \approx \frac{L(\theta_i + h) - L(\theta_i - h)}{2h}$$

Hai điểm kỹ thuật quan trọng:

1. **ReLU và MaxPool làm hàm mất mát có "nếp gấp"**. Nếu phép nhiễu $\pm h$ đổi
   dấu một pre-activation hay đổi ô thắng của pooling thì sai phân số đi xuyên
   qua điểm không khả vi và không còn xấp xỉ đạo hàm. Ta **phát hiện và loại
   riêng** những toạ độ đó thay vì nới lỏng ngưỡng cho toàn bộ phép kiểm tra.
2. **Hàm mất mát phải nhất quán với gradient.** Cách viết `-log(p + eps)` gài
   một sai số hệ thống cỡ $\epsilon/p$; khi $p \sim 10^{-8}$ sai số này lên tới
   $10^{-4}$ và phép kiểm tra sẽ thất bại dù backward hoàn toàn đúng. Vì vậy
   `loss()` tính từ **logits** bằng dạng log-sum-exp, không cần `eps` — đúng
   lý do PyTorch khuyên dùng `binary_cross_entropy_with_logits`.

In [13]:
# ============================================================================
# KHỐI 12 — Gradient check trên chính kiến trúc sẽ dùng để huấn luyện
# ============================================================================
def pool_relu_pattern(net):
    """Ảnh chụp 'hình dạng tuyến tính hoá' (dấu ReLU + ô thắng pooling)."""
    pat = []
    for L in net.layers:
        if isinstance(L, (MaxPool1D, GlobalMaxPool1D)):
            pat.append(L._argmax.copy())
        elif isinstance(L, ReLU):
            pat.append(L._z > 0)
    return pat


def same_pattern(a, b):
    return len(a) == len(b) and all(np.array_equal(u, v) for u, v in zip(a, b))


def gradient_check(net, X, yy, n_coord=30, h=1e-5):
    p, z = net.forward(X, training=False)
    base = pool_relu_pattern(net)
    net.backward(p, yy)
    grads = [g.copy() for L in net.layers for g in L.grad_list()]
    params = [q for L in net.layers for q in L.param_list()]

    worst, tested, skipped = 0.0, 0, 0
    for q, g in zip(params, grads):
        flat = q.ravel()
        for i in np.linspace(0, flat.size - 1, min(n_coord, flat.size)).astype(int):
            old = flat[i]
            flat[i] = old + h
            pp, zp = net.forward(X, training=False)
            lp, pat_p = net.loss(pp, yy, logits=zp), pool_relu_pattern(net)
            flat[i] = old - h
            pm, zm = net.forward(X, training=False)
            lm, pat_m = net.loss(pm, yy, logits=zm), pool_relu_pattern(net)
            flat[i] = old
            if not (same_pattern(base, pat_p) and same_pattern(base, pat_m)):
                skipped += 1
                continue
            num = (lp - lm) / (2 * h)
            ana = g.ravel()[i]
            worst = max(worst, abs(num - ana) / max(1e-8, abs(num) + abs(ana)))
            tested += 1
    return worst, tested, skipped


check_net = build_cnn(seed=7)
worst, tested, skipped = gradient_check(check_net, Xtr[:24], y_train[:24])
print(f"Sai số tương đối lớn nhất : {worst:.3e}")
print(f"Số toạ độ đã kiểm tra     : {tested}")
print(f"Số toạ độ bị loại (nếp gấp ReLU/MaxPool): {skipped}")
assert worst < 1e-6, "Gradient check FAILED"
print("\n✅ Gradient viết tay khớp gradient sai phân số — bản backward là đúng.")

Sai số tương đối lớn nhất : 4.229e-07
Số toạ độ đã kiểm tra     : 215
Số toạ độ bị loại (nếp gấp ReLU/MaxPool): 0

✅ Gradient viết tay khớp gradient sai phân số — bản backward là đúng.


## 6. Huấn luyện bản NumPy from scratch

In [14]:
# ============================================================================
# KHỐI 13 — Huấn luyện CNN bằng NumPy, early stopping theo val_loss
# ============================================================================
cnn = build_cnn()
t0 = time.time()
cnn.fit(Xtr, y_train, Xva, y_val,
        epochs=HP["epochs"], batch_size=HP["batch_size"],
        patience=HP["patience"], verbose_every=25)
numpy_time = time.time() - t0
print(f"\n⏱ NumPy from scratch: {numpy_time:.2f}s "
      f"({len(cnn.history['train_loss'])} epoch đã chạy)")

  epoch    1 | train_loss=0.6183 | train_metric=0.6884 | val_loss=0.5947 | val_metric=0.7069


  epoch   25 | train_loss=0.3419 | train_metric=0.8507 | val_loss=0.5847 | val_metric=0.7328


  epoch   50 | train_loss=0.2334 | train_metric=0.8993 | val_loss=0.7126 | val_metric=0.7586


  ⏹ Early stopping tại epoch 64 (val_loss tốt nhất = 0.5202)

⏱ NumPy from scratch: 2.06s (64 epoch đã chạy)


## 7. Bản PyTorch tương đương

Cùng kiến trúc, cùng seed, cùng optimizer Adam, cùng siêu tham số. Mục đích là
kiểm chứng bản tự cài cho kết quả ngang với framework chuẩn. Điểm khác biệt duy
nhất về mặt thư viện: PyTorch dùng `nn.Conv1d` với autograd thay cho
`conv1d_forward`/`conv1d_backward` viết tay.

In [15]:
# ============================================================================
# KHỐI 14 — Dựng và huấn luyện bản PyTorch
# Lưu ý layout: PyTorch dùng channels-first (N, C, L) — giống hệt bản NumPy,
# nên Xtr đưa vào trực tiếp không cần transpose.
# ============================================================================
import torch
import torch.nn as nn

torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True)

D = HP["dropout"]
torch_net = nn.Sequential(
    nn.Conv1d(1, 16, 3), nn.ReLU(),
    nn.Conv1d(16, 16, 3), nn.ReLU(),
    nn.MaxPool1d(2),
    nn.Conv1d(16, 32, 2), nn.ReLU(),
    nn.Flatten(),
    nn.Dropout(D),
    nn.Linear(32, 16), nn.ReLU(),
    nn.Linear(16, 1),
)
print(torch_net)
n_torch = sum(p.numel() for p in torch_net.parameters())
print(f"\nTổng tham số PyTorch : {n_torch:,}")
print(f"Tổng tham số NumPy   : {cnn.n_params():,}")
assert n_torch == cnn.n_params(), "Hai bản không cùng số tham số!"
print("✅ Hai bản có đúng cùng số tham số.")

Sequential(
  (0): Conv1d(1, 16, kernel_size=(3,), stride=(1,))
  (1): ReLU()
  (2): Conv1d(16, 16, kernel_size=(3,), stride=(1,))
  (3): ReLU()
  (4): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (5): Conv1d(16, 32, kernel_size=(2,), stride=(1,))
  (6): ReLU()
  (7): Flatten(start_dim=1, end_dim=-1)
  (8): Dropout(p=0.3, inplace=False)
  (9): Linear(in_features=32, out_features=16, bias=True)
  (10): ReLU()
  (11): Linear(in_features=16, out_features=1, bias=True)
)

Tổng tham số PyTorch : 2,449
Tổng tham số NumPy   : 2,449
✅ Hai bản có đúng cùng số tham số.


In [16]:
# ============================================================================
# KHỐI 15 — Vòng huấn luyện PyTorch, giữ nguyên early stopping theo val_loss
# BCEWithLogitsLoss = Sigmoid + BCE gộp lại, chính là dạng logit mà bản NumPy
# dùng trong loss() — nên hai bản tối ưu đúng cùng một hàm mục tiêu.
# ============================================================================
Xtr_t = torch.tensor(Xtr, dtype=torch.float32)
ytr_t = torch.tensor(y_train, dtype=torch.float32)
Xva_t = torch.tensor(Xva, dtype=torch.float32)
yva_t = torch.tensor(y_val, dtype=torch.float32)
Xte_t = torch.tensor(Xte, dtype=torch.float32)

opt = torch.optim.Adam(torch_net.parameters(), lr=HP["lr"], weight_decay=0.0)
lossf = nn.BCEWithLogitsLoss()
torch_hist = {"train_loss": [], "val_loss": []}

g = torch.Generator().manual_seed(SEED)
best_vl, best_sd, wait = np.inf, None, 0
t0 = time.time()
for ep in range(1, HP["epochs"] + 1):
    torch_net.train()
    perm = torch.randperm(len(Xtr_t), generator=g)
    for s in range(0, len(perm), HP["batch_size"]):
        sl = perm[s:s + HP["batch_size"]]
        opt.zero_grad()
        out = torch_net(Xtr_t[sl])
        # L2 thêm tay để khớp đúng công thức l2*Σw²/(2n) của bản NumPy
        l2_pen = sum((p ** 2).sum() for p in torch_net.parameters())
        loss = lossf(out, ytr_t[sl]) + HP["l2"] * l2_pen / (2 * len(sl))
        loss.backward()
        opt.step()

    torch_net.eval()
    with torch.no_grad():
        tl = float(lossf(torch_net(Xtr_t), ytr_t))
        vl = float(lossf(torch_net(Xva_t), yva_t))
    torch_hist["train_loss"].append(tl)
    torch_hist["val_loss"].append(vl)

    if vl < best_vl - 1e-6:
        best_vl, wait = vl, 0
        best_sd = {k: v.clone() for k, v in torch_net.state_dict().items()}
    else:
        wait += 1
        if wait >= HP["patience"]:
            print(f"  ⏹ Early stopping tại epoch {ep} (val_loss tốt nhất = {best_vl:.4f})")
            break
    if ep % 25 == 0 or ep == 1:
        print(f"  epoch {ep:4d} | train_loss={tl:.4f} | val_loss={vl:.4f}")

if best_sd is not None:
    torch_net.load_state_dict(best_sd)
torch_time = time.time() - t0
print(f"\n⏱ PyTorch: {torch_time:.2f}s ({len(torch_hist['train_loss'])} epoch)")

  epoch    1 | train_loss=0.6737 | val_loss=0.6743


  epoch   25 | train_loss=0.3969 | val_loss=0.5155


  epoch   50 | train_loss=0.3108 | val_loss=0.6390


  ⏹ Early stopping tại epoch 71 (val_loss tốt nhất = 0.4970)

⏱ PyTorch: 7.04s (71 epoch)


## 8. Bản TensorFlow/Keras tương đương

Cùng một ý tưởng toán học, cú pháp khác. Điểm cần lưu ý nhất:
**Keras dùng channels-last** — `Conv1D` mong đợi `(N, L, C)` thay vì `(N, C, L)`.
Vì thế ta phải transpose dữ liệu. Đây chính là một trong các nguồn sai khác
giữa các nền tảng được phân tích ở Chương IV.

In [17]:
# ============================================================================
# KHỐI 16 — Dựng và huấn luyện bản TensorFlow/Keras
# Transpose (N, C, L) -> (N, L, C) cho channels-last.
# ============================================================================
import tensorflow as tf
from tensorflow import keras

tf.keras.utils.set_random_seed(SEED)

Xtr_tf = np.transpose(Xtr, (0, 2, 1))     # (N, 8, 1)
Xva_tf = np.transpose(Xva, (0, 2, 1))
Xte_tf = np.transpose(Xte, (0, 2, 1))
print("channels-first (NumPy/PyTorch):", Xtr.shape)
print("channels-last  (TensorFlow)   :", Xtr_tf.shape)

reg = keras.regularizers.l2(HP["l2"] / 2)
tf_net = keras.Sequential([
    keras.layers.Input(shape=(8, 1)),
    keras.layers.Conv1D(16, 3, activation="relu", kernel_regularizer=reg),
    keras.layers.Conv1D(16, 3, activation="relu", kernel_regularizer=reg),
    keras.layers.MaxPooling1D(2),
    keras.layers.Conv1D(32, 2, activation="relu", kernel_regularizer=reg),
    keras.layers.Flatten(),
    keras.layers.Dropout(D),
    keras.layers.Dense(16, activation="relu", kernel_regularizer=reg),
    keras.layers.Dense(1, activation=None),
])
tf_net.summary()
n_tf = tf_net.count_params()
print(f"\nTổng tham số TensorFlow: {n_tf:,}")
assert n_tf == cnn.n_params(), "Bản TF không cùng số tham số!"
print("✅ Ba bản NumPy / PyTorch / TensorFlow có đúng cùng số tham số.")

channels-first (NumPy/PyTorch): (536, 1, 8)
channels-last  (TensorFlow)   : (536, 8, 1)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 6, 16)          │            64 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 4, 16)          │           784 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 2, 16)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 1, 32)          │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,449 (9.57 KB)

 Trainable params: 2,449 (9.57 KB)

 Non-trainable params: 0 (0.00 B)


Tổng tham số TensorFlow: 2,449
✅ Ba bản NumPy / PyTorch / TensorFlow có đúng cùng số tham số.


In [18]:
# ============================================================================
# KHỐI 17 — Huấn luyện bản TensorFlow bằng fit() + EarlyStopping callback
# from_logits=True để khớp với hai bản kia (Sigmoid gộp trong loss).
# ============================================================================
tf_net.compile(
    optimizer=keras.optimizers.Adam(learning_rate=HP["lr"]),
    loss=keras.losses.BinaryCrossentropy(from_logits=True),
    metrics=[keras.metrics.AUC(name="auc", from_logits=True)],
)
es = keras.callbacks.EarlyStopping(monitor="val_loss", patience=HP["patience"],
                                   restore_best_weights=True, verbose=0)
t0 = time.time()
tf_hist = tf_net.fit(
    Xtr_tf, y_train,
    validation_data=(Xva_tf, y_val),
    epochs=HP["epochs"], batch_size=HP["batch_size"],
    callbacks=[es], verbose=0,
)
tf_time = time.time() - t0
print(f"⏱ TensorFlow: {tf_time:.2f}s ({len(tf_hist.history['loss'])} epoch)")
print(f"  val_loss tốt nhất = {min(tf_hist.history['val_loss']):.4f}")

⏱ TensorFlow: 16.80s (84 epoch)
  val_loss tốt nhất = 0.5662


## 9. Các mô hình đối sánh

Để kết luận có ý nghĩa, CNN phải được so với:

- **MLP 5 tầng của Assignment 03** — cùng độ sâu, nhưng kết nối đầy đủ. Đây là
  phép so sánh quan trọng nhất: nó tách riêng đóng góp của *tích chập* khỏi
  đóng góp của *độ sâu*.
- **Học máy truyền thống** — Logistic Regression, Decision Tree, Random Forest.

In [19]:
# ============================================================================
# KHỐI 18 — MLP 5 tầng (Assignment 03) trên cùng dữ liệu phẳng, cùng seed
# Cài lại ngắn gọn bằng PyTorch cho gọn; kiến trúc 8-128-64-32-16-1 như A03.
# ============================================================================
torch.manual_seed(SEED)
mlp_net = nn.Sequential(
    nn.Linear(8, 128), nn.ReLU(), nn.Dropout(0.40),
    nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.40),
    nn.Linear(64, 32), nn.ReLU(), nn.Dropout(0.40),
    nn.Linear(32, 16), nn.ReLU(), nn.Dropout(0.40),
    nn.Linear(16, 1),
)
Xtr_mlp = torch.tensor(Xtr_flat, dtype=torch.float32)
Xva_mlp = torch.tensor(Xva_flat, dtype=torch.float32)
Xte_mlp = torch.tensor(Xte_flat, dtype=torch.float32)

opt_m = torch.optim.Adam(mlp_net.parameters(), lr=1e-3)
gm = torch.Generator().manual_seed(SEED)
best_vl_m, best_sd_m, wait_m = np.inf, None, 0
for ep in range(1, 401):
    mlp_net.train()
    perm = torch.randperm(len(Xtr_mlp), generator=gm)
    for s in range(0, len(perm), 32):
        sl = perm[s:s + 32]
        opt_m.zero_grad()
        l2p = sum((p ** 2).sum() for p in mlp_net.parameters())
        loss = lossf(mlp_net(Xtr_mlp[sl]), ytr_t[sl]) + 5e-2 * l2p / (2 * len(sl))
        loss.backward()
        opt_m.step()
    mlp_net.eval()
    with torch.no_grad():
        vl = float(lossf(mlp_net(Xva_mlp), yva_t))
    if vl < best_vl_m - 1e-6:
        best_vl_m, wait_m = vl, 0
        best_sd_m = {k: v.clone() for k, v in mlp_net.state_dict().items()}
    else:
        wait_m += 1
        if wait_m >= 60:
            break
if best_sd_m is not None:
    mlp_net.load_state_dict(best_sd_m)
print(f"MLP-5 (Assignment 03): {sum(p.numel() for p in mlp_net.parameters()):,} tham số, "
      f"dừng ở epoch {ep}, val_loss = {best_vl_m:.4f}")

MLP-5 (Assignment 03): 12,033 tham số, dừng ở epoch 84, val_loss = 0.4705


In [20]:
# ============================================================================
# KHỐI 19 — Ba mô hình Học máy truyền thống trên dữ liệu phẳng
# ============================================================================
classical = {
    "Logistic Regression": LogisticRegression(max_iter=2000, random_state=SEED),
    "Decision Tree": DecisionTreeClassifier(max_depth=5, random_state=SEED),
    "Random Forest": RandomForestClassifier(n_estimators=200, max_depth=6,
                                            random_state=SEED, n_jobs=-1),
}
for name, mdl in classical.items():
    mdl.fit(Xtr_flat, y_train.ravel())
    print(f"  {name:<22} val ROC-AUC = "
          f"{roc_auc_score(y_val, mdl.predict_proba(Xva_flat)[:, 1]):.4f}")

  Logistic Regression    val ROC-AUC = 0.8367
  Decision Tree          val ROC-AUC = 0.7041


  Random Forest          val ROC-AUC = 0.8286


## 10. Đánh giá trên tập kiểm thử

In [21]:
# ============================================================================
# KHỐI 20 — Thu xác suất dự đoán của mọi mô hình trên tập TEST
# Ba bản CNN dùng chung một hàm đánh giá để bảo đảm so sánh công bằng.
# ============================================================================
def sigmoid_np(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -60, 60)))


proba = {}
proba["CNN-5 (NumPy)"] = cnn.predict_proba(Xte).ravel()

torch_net.eval()
with torch.no_grad():
    proba["CNN-5 (PyTorch)"] = sigmoid_np(torch_net(Xte_t).numpy().ravel())
    proba["MLP-5 (A03, PyTorch)"] = sigmoid_np(mlp_net(Xte_mlp).numpy().ravel())

proba["CNN-5 (TensorFlow)"] = sigmoid_np(tf_net.predict(Xte_tf, verbose=0).ravel())

for name, mdl in classical.items():
    proba[name] = mdl.predict_proba(Xte_flat)[:, 1]

yte = y_test.ravel()


def evaluate(p, thr=0.5):
    pred = (p >= thr).astype(int)
    return {
        "Accuracy": accuracy_score(yte, pred),
        "Precision": precision_score(yte, pred, zero_division=0),
        "Recall": recall_score(yte, pred, zero_division=0),
        "F1": f1_score(yte, pred, zero_division=0),
        "ROC-AUC": roc_auc_score(yte, p),
    }


res_df = pd.DataFrame({k: evaluate(v) for k, v in proba.items()}).T
res_df = res_df.sort_values("ROC-AUC", ascending=False)
print("Kết quả trên tập TEST (ngưỡng mặc định 0.5):\n")
res_df.round(4)

Kết quả trên tập TEST (ngưỡng mặc định 0.5):



,Accuracy,Precision,Recall,F1,ROC-AUC
"MLP-5 (A03, PyTorch)",0.7328,0.6154,0.600,0.6076,0.8391
CNN-5 (TensorFlow),0.7241,0.5952,0.625,0.6098,0.8368
CNN-5 (NumPy),0.7500,0.6774,0.525,0.5915,0.8355
Logistic Regression,0.7069,0.5789,0.550,0.5641,0.8345
CNN-5 (PyTorch),0.7328,0.6154,0.600,0.6076,0.8336
Random Forest,0.7672,0.6857,0.600,0.6400,0.8250
Decision Tree,0.7328,0.6452,0.500,0.5634,0.7752


### 10.1. Ba nền tảng có cho cùng kết quả không?

Đây là câu hỏi cốt lõi của Assignment 04. Ba bản cài cùng một mô hình toán học
nên **về lý thuyết** phải cho kết quả như nhau, nhưng trên thực tế luôn lệch
nhẹ. Bảng dưới đo mức lệch đó.

In [22]:
# ============================================================================
# KHỐI 21 — Đo độ lệch giữa ba nền tảng trên cùng tập test
# ============================================================================
tri = ["CNN-5 (NumPy)", "CNN-5 (PyTorch)", "CNN-5 (TensorFlow)"]
pairs = [(tri[0], tri[1]), (tri[0], tri[2]), (tri[1], tri[2])]
rows = []
for a, b in pairs:
    pa, pb = proba[a], proba[b]
    rows.append({
        "Cặp so sánh": f"{a.split('(')[1][:-1]} ↔ {b.split('(')[1][:-1]}",
        "Sai khác xác suất TB": float(np.mean(np.abs(pa - pb))),
        "Sai khác lớn nhất": float(np.abs(pa - pb).max()),
        "Tương quan Pearson": float(np.corrcoef(pa, pb)[0, 1]),
        "Số dự đoán khác nhau": int(np.sum((pa >= 0.5) != (pb >= 0.5))),
    })
divergence = pd.DataFrame(rows)
print(f"Tập test có {len(yte)} mẫu.\n")
print("Ba nền tảng KHÔNG cho kết quả trùng khít từng chữ số, vì:")
print("  • thứ tự cộng dồn dấu phẩy động khác nhau (BLAS vs einsum vs oneDNN);")
print("  • bộ sinh số giả ngẫu nhiên khác nhau -> khởi tạo và dropout khác nhau;")
print("  • thứ tự trộn mini-batch khác nhau.\n")
divergence.round(6)

Tập test có 116 mẫu.

Ba nền tảng KHÔNG cho kết quả trùng khít từng chữ số, vì:
  • thứ tự cộng dồn dấu phẩy động khác nhau (BLAS vs einsum vs oneDNN);
  • bộ sinh số giả ngẫu nhiên khác nhau -> khởi tạo và dropout khác nhau;
  • thứ tự trộn mini-batch khác nhau.



,Cặp so sánh,Sai khác xác suất TB,Sai khác lớn nhất,Tương quan Pearson,Số dự đoán khác nhau
0,NumPy ↔ PyTorch,0.109116,0.518146,0.851505,18
1,NumPy ↔ TensorFlow,0.110418,0.501016,0.870391,15
2,PyTorch ↔ TensorFlow,0.053975,0.293334,0.983400,5


### 10.2. Hiệu chỉnh ngưỡng quyết định

Trong chẩn đoán y khoa, **bỏ sót người bệnh (FN) tốn kém hơn** báo động sai
(FP). Ngưỡng 0.5 tối ưu cho Accuracy chứ không tối ưu cho F1. Ta chọn ngưỡng
theo F1 **trên tập validation**, rồi mới áp lên tập test.

In [23]:
# ============================================================================
# KHỐI 22 — Chọn ngưỡng trên VALIDATION rồi áp lên TEST
# Đây là điểm dễ sai: nếu chọn ngưỡng trên chính tập test thì đã rò rỉ dữ liệu.
# ============================================================================
proba_val = {}
proba_val["CNN-5 (NumPy)"] = cnn.predict_proba(Xva).ravel()
with torch.no_grad():
    proba_val["CNN-5 (PyTorch)"] = sigmoid_np(torch_net(Xva_t).numpy().ravel())
    proba_val["MLP-5 (A03, PyTorch)"] = sigmoid_np(mlp_net(Xva_mlp).numpy().ravel())
proba_val["CNN-5 (TensorFlow)"] = sigmoid_np(tf_net.predict(Xva_tf, verbose=0).ravel())
for name, mdl in classical.items():
    proba_val[name] = mdl.predict_proba(Xva_flat)[:, 1]

grid = np.linspace(0.05, 0.95, 91)
best_thr = {}
for name, pv in proba_val.items():
    f1s = [f1_score(y_val.ravel(), (pv >= t).astype(int), zero_division=0) for t in grid]
    best_thr[name] = float(grid[int(np.argmax(f1s))])

tuned_df = pd.DataFrame(
    {k: evaluate(proba[k], best_thr[k]) for k in proba}).T
tuned_df["Ngưỡng"] = [best_thr[k] for k in tuned_df.index]
tuned_df = tuned_df.loc[res_df.index]
print("Kết quả sau khi hiệu chỉnh ngưỡng (ngưỡng chọn trên VALIDATION):\n")
tuned_df.round(4)

Kết quả sau khi hiệu chỉnh ngưỡng (ngưỡng chọn trên VALIDATION):



,Accuracy,Precision,Recall,F1,ROC-AUC,Ngưỡng
"MLP-5 (A03, PyTorch)",0.7845,0.6596,0.775,0.7126,0.8391,0.39
CNN-5 (TensorFlow),0.7155,0.5714,0.700,0.6292,0.8368,0.43
CNN-5 (NumPy),0.7500,0.6486,0.600,0.6234,0.8355,0.48
Logistic Regression,0.7500,0.5902,0.900,0.7129,0.8345,0.24
CNN-5 (PyTorch),0.7414,0.6190,0.650,0.6341,0.8336,0.40
Random Forest,0.7155,0.5714,0.700,0.6292,0.8250,0.39
Decision Tree,0.7241,0.5714,0.800,0.6667,0.7752,0.24


## 11. Thí nghiệm thứ tự cột — kiểm chứng Cảnh báo Mô hình hoá

Slide 8 cảnh báo: cột trong file CSV không có tính cục bộ không gian như điểm
ảnh. Câu hỏi cần trả lời bằng thực nghiệm: **CNN có thực sự khai thác quan hệ
giữa các cột liền kề, hay thứ tự cột chỉ ảnh hưởng qua kiến trúc?**

Một phép thử "xáo trộn rồi xem có tệ đi không" là **chưa đủ**, vì hoán vị ngẫu
nhiên thay đổi *hai* thứ cùng lúc:

1. **quan hệ liền kề** — cột nào nằm cạnh cột nào;
2. **vị trí tuyệt đối** — với tích chập 'valid' + pooling, cột ở giữa chuỗi
   tham gia nhiều cửa sổ hơn cột ở hai đầu, nên vị trí tự nó đã quan trọng.

Vì vậy ta thiết kế **bốn nghiệm thức**:

| # | Nghiệm thức | Quan hệ liền kề | Độ phủ cửa sổ |
|---|---|---|---|
| 1 | Thứ tự gốc | giữ nguyên | giữ nguyên |
| 2 | **Đảo ngược** | giữ nguyên | giữ nguyên *(xem ghi chú)* |
| 3 | Hoán vị ngẫu nhiên (n = 10) | bị phá | đổi |
| 4 | **Quét vị trí `Glucose`** | gần như giữ | đổi có kiểm soát |

> **Ghi chú quan trọng về nghiệm thức 2.** Đảo ngược *không* phải là phép thử
> tách biến như thoạt nhìn: nó là một **phép đối xứng của chính kiến trúc**.
> Vì hồ sơ độ phủ $[1,2,3,3,3,3,2,1]$ đối xứng qua tâm, đảo ngược giữ nguyên
> *cả* quan hệ liền kề *lẫn* độ phủ của từng cột (cột thứ 2 từ đầu đổi chỗ cho
> cột thứ 2 từ cuối — cùng độ phủ). Do đó ta **dự đoán trước** nó cho kết quả
> gần như trùng khít, và nếu quan sát đúng như vậy thì đó là một phép **kiểm
> chứng tính đúng đắn của cài đặt**, chứ không phải bằng chứng về tính cục bộ.

Nghiệm thức 4 mới là phép thử quyết định: giữ nguyên thứ tự tương đối của 7 cột
còn lại và chỉ **dịch `Glucose` — đặc trưng phân tách mạnh nhất — qua đủ 8 vị
trí**. Nếu hiệu năng bám theo *độ phủ cửa sổ* của vị trí đó thì yếu tố quyết
định là **vị trí**, không phải **quan hệ liền kề**.

Ta cũng chạy **Logistic Regression** làm đối chứng: mô hình này **bất biến
chính xác** với hoán vị cột, nên nó xác nhận bản thân quy trình đánh giá không
tự sinh ra biến động nào.

In [24]:
# ============================================================================
# KHỐI 23 — Ba nghiệm thức: thứ tự gốc, đảo ngược, và 10 hoán vị ngẫu nhiên
# Mọi nghiệm thức dùng CÙNG seed khởi tạo và cùng siêu tham số; chỉ thứ tự cột
# đổi. Nhờ vậy chênh lệch quan sát được quy hết về thứ tự cột.
# ============================================================================
N_PERM = 10
rng_perm = np.random.default_rng(SEED)


def train_cnn_with_order(order):
    """Huấn luyện lại CNN với một thứ tự cột cho trước, trả về ROC-AUC test."""
    Xtr_p = Xtr_flat[:, order][:, None, :]
    Xva_p = Xva_flat[:, order][:, None, :]
    Xte_p = Xte_flat[:, order][:, None, :]
    net_p = build_cnn(seed=SEED)
    net_p.fit(Xtr_p, y_train, Xva_p, y_val, epochs=HP["epochs"],
              batch_size=HP["batch_size"], patience=HP["patience"], verbose_every=0)
    return float(roc_auc_score(yte, net_p.predict_proba(Xte_p).ravel()))


base_auc = float(res_df.loc["CNN-5 (NumPy)", "ROC-AUC"])
identity = np.arange(8)

# --- Hồ sơ "độ phủ": mỗi vị trí đầu vào nối tới đầu ra qua bao nhiêu đường? ---
# Đếm bằng một mạng thế thân toàn số 1 (không ReLU, pooling thay bằng tổng), cho
# đầu vào one-hot ở từng vị trí. Con số thu được chính là số đường đi từ ô đó
# tới đầu ra của Conv3 — thước đo khách quan cho "cột này được nhìn thấy bao nhiêu".
def coverage_profile(n_in=8):
    cov = []
    ones1 = np.ones((1, 1, 3)); ones2 = np.ones((1, 1, 3)); ones3 = np.ones((1, 1, 2))
    z0 = np.array([0.0])
    for i in range(n_in):
        x = np.zeros((1, 1, n_in))
        x[0, 0, i] = 1.0
        a = conv1d_forward(x, ones1, z0)[0]              # (1,1,6)
        a = conv1d_forward(a, ones2, z0)[0]              # (1,1,4)
        a = a.reshape(1, 1, 2, 2).sum(axis=3)            # pooling -> tổng (đếm đường)
        a = conv1d_forward(a, ones3, z0)[0]              # (1,1,1)
        cov.append(float(a.sum()))
    return np.array(cov)


COVER = coverage_profile(8)
print("Độ phủ của từng vị trí trong chuỗi (số đường đi tới đầu ra):")
print(f"  vị trí : {list(range(8))}")
print(f"  độ phủ : {COVER.astype(int).tolist()}")
print(f"  -> hồ sơ ĐỐI XỨNG qua tâm, nên đảo ngược không đổi độ phủ của bất kỳ cột nào.")
glu_pos = FEATURES.index("Glucose")
print(f"  Glucose nằm ở vị trí {glu_pos} (độ phủ {COVER[glu_pos]:.0f})\n")

# (1) Đảo ngược — phép đối xứng của kiến trúc, dùng để KIỂM CHỨNG cài đặt
rev_auc = train_cnn_with_order(identity[::-1].copy())
print(f"Thứ tự gốc      : ROC-AUC = {base_auc:.4f}")
print(f"Thứ tự đảo ngược: ROC-AUC = {rev_auc:.4f}   "
      f"(chênh {rev_auc - base_auc:+.6f} — dự đoán ≈ 0 vì đây là phép đối xứng)\n")

# (2) Hoán vị ngẫu nhiên — phá quan hệ liền kề
perm_aucs, perm_orders = [], []
for trial in range(N_PERM):
    order = rng_perm.permutation(8)
    auc_p = train_cnn_with_order(order)
    perm_aucs.append(auc_p)
    perm_orders.append(order.tolist())
    print(f"  hoán vị #{trial + 1:2d} {order.tolist()} -> ROC-AUC = {auc_p:.4f}")

perm_aucs = np.array(perm_aucs)

# (3) Quét vị trí của Glucose — giữ thứ tự tương đối của 7 cột còn lại
others = [i for i in range(8) if i != glu_pos]
sweep_aucs = []
print()
for pos in range(8):
    order = others[:pos] + [glu_pos] + others[pos:]
    auc_s = train_cnn_with_order(np.array(order))
    sweep_aucs.append(auc_s)
    print(f"  Glucose ở vị trí {pos} (độ phủ {COVER[pos]:.0f}) -> ROC-AUC = {auc_s:.4f}")
sweep_aucs = np.array(sweep_aucs)

# (4) Đối chứng: Logistic Regression bất biến chính xác với hoán vị cột
lr_aucs = []
for order in perm_orders:
    lr = LogisticRegression(max_iter=2000, random_state=SEED)
    lr.fit(Xtr_flat[:, order], y_train.ravel())
    lr_aucs.append(float(roc_auc_score(yte, lr.predict_proba(Xte_flat[:, order])[:, 1])))
lr_aucs = np.array(lr_aucs)

perm_df = pd.DataFrame(
    [{"Nghiệm thức": "Thứ tự gốc (file CSV)", "Quan hệ liền kề": "giữ nguyên",
      "Độ phủ": "giữ nguyên", "ROC-AUC test": base_auc},
     {"Nghiệm thức": "Đảo ngược (phép đối xứng)", "Quan hệ liền kề": "giữ nguyên",
      "Độ phủ": "giữ nguyên", "ROC-AUC test": rev_auc}]
    + [{"Nghiệm thức": f"Hoán vị #{i+1}", "Quan hệ liền kề": "bị phá",
        "Độ phủ": "đổi", "ROC-AUC test": a} for i, a in enumerate(perm_aucs)]
    + [{"Nghiệm thức": f"Glucose ở vị trí {i}", "Quan hệ liền kề": "gần như giữ",
        "Độ phủ": f"{COVER[i]:.0f}", "ROC-AUC test": a}
       for i, a in enumerate(sweep_aucs)])

print(f"\n{'':-<70}")
print(f"Gốc                          : {base_auc:.4f}")
print(f"Đảo ngược                    : {rev_auc:.4f}  (chênh {rev_auc - base_auc:+.6f})")
print(f"Hoán vị ngẫu nhiên (n={N_PERM})   : {perm_aucs.mean():.4f} ± {perm_aucs.std(ddof=1):.4f}"
      f"  [{perm_aucs.min():.4f}, {perm_aucs.max():.4f}]")
print(f"Quét vị trí Glucose          : {sweep_aucs.mean():.4f} ± {sweep_aucs.std(ddof=1):.4f}"
      f"  [{sweep_aucs.min():.4f}, {sweep_aucs.max():.4f}]")
print(f"Đối chứng LogisticRegression : {lr_aucs.mean():.4f} ± {lr_aucs.std(ddof=1):.2e}"
      f"  (bất biến — đúng lý thuyết)")
print(f"{'':-<70}")
perm_df.round(4)

Độ phủ của từng vị trí trong chuỗi (số đường đi tới đầu ra):
  vị trí : [0, 1, 2, 3, 4, 5, 6, 7]
  độ phủ : [1, 3, 6, 8, 8, 6, 3, 1]
  -> hồ sơ ĐỐI XỨNG qua tâm, nên đảo ngược không đổi độ phủ của bất kỳ cột nào.
  Glucose nằm ở vị trí 1 (độ phủ 3)



Thứ tự gốc      : ROC-AUC = 0.8355
Thứ tự đảo ngược: ROC-AUC = 0.8355   (chênh +0.000000 — dự đoán ≈ 0 vì đây là phép đối xứng)



  hoán vị # 1 [3, 4, 2, 7, 6, 1, 5, 0] -> ROC-AUC = 0.8118


  hoán vị # 2 [0, 2, 7, 1, 4, 5, 3, 6] -> ROC-AUC = 0.7885


  hoán vị # 3 [5, 7, 2, 4, 1, 6, 3, 0] -> ROC-AUC = 0.8115


...

  Glucose ở vị trí 4 (độ phủ 8) -> ROC-AUC = 0.7970


  Glucose ở vị trí 5 (độ phủ 6) -> ROC-AUC = 0.8359


  Glucose ở vị trí 6 (độ phủ 3) -> ROC-AUC = 0.8401


  Glucose ở vị trí 7 (độ phủ 1) -> ROC-AUC = 0.7967

----------------------------------------------------------------------
Gốc                          : 0.8355
Đảo ngược                    : 0.8355  (chênh +0.000000)
Hoán vị ngẫu nhiên (n=10)   : 0.7936 ± 0.0172  [0.7655, 0.8234]
Quét vị trí Glucose          : 0.8282 ± 0.0217  [0.7967, 0.8523]
Đối chứng LogisticRegression : 0.8345 ± 1.17e-16  (bất biến — đúng lý thuyết)
----------------------------------------------------------------------


,Nghiệm thức,Quan hệ liền kề,Độ phủ,ROC-AUC test
0,Thứ tự gốc (file CSV),giữ nguyên,giữ nguyên,0.8355
1,Đảo ngược (phép đối xứng),giữ nguyên,giữ nguyên,0.8355
2,Hoán vị #1,bị phá,đổi,0.8118
3,Hoán vị #2,bị phá,đổi,0.7885
4,Hoán vị #3,bị phá,đổi,0.8115
5,Hoán vị #4,bị phá,đổi,0.7655
6,Hoán vị #5,bị phá,đổi,0.7931
7,Hoán vị #6,bị phá,đổi,0.8234
8,Hoán vị #7,bị phá,đổi,0.7812
9,Hoán vị #8,bị phá,đổi,0.7832


### 11.1. Đọc kết quả thí nghiệm

Mọi kết luận dưới đây được **sinh tự động từ số liệu vừa chạy**, không viết sẵn.

In [25]:
# ============================================================================
# KHỐI 23b — Diễn giải dựa trên chính số liệu vừa đo
# Câu hỏi: biến động ROC-AUC giải thích được bằng ĐỘ PHỦ vị trí hay không?
# ============================================================================
drop_perm = base_auc - perm_aucs.mean()
drop_rev = base_auc - rev_auc
spread = perm_aucs.std(ddof=1)

# Tương quan giữa ROC-AUC và độ phủ của vị trí Glucose, trên nghiệm thức quét
r_sweep = float(np.corrcoef(COVER, sweep_aucs)[0, 1])
# Và trên 10 hoán vị ngẫu nhiên: độ phủ của vị trí mà Glucose rơi vào
glu_cov_perm = np.array([COVER[list(o).index(glu_pos)] for o in perm_orders])
r_perm = float(np.corrcoef(glu_cov_perm, perm_aucs)[0, 1])

print("QUAN SÁT")
print(f"  1. Đảo ngược lệch {rev_auc - base_auc:+.6f} so với gốc.")
print(f"     -> đúng như dự đoán lý thuyết: đây là phép ĐỐI XỨNG của kiến trúc")
print(f"        (hồ sơ độ phủ {COVER.astype(int).tolist()} đối xứng qua tâm),")
print(f"        nên nó KIỂM CHỨNG cài đặt chứ không tách được biến nào.")
print(f"  2. Hoán vị ngẫu nhiên giảm trung bình {drop_perm:+.4f} "
      f"({drop_perm / spread:.1f} lần độ lệch chuẩn); {int(np.sum(perm_aucs >= base_auc))}/{N_PERM} "
      f"hoán vị đạt >= gốc.")
print(f"  3. Quét vị trí Glucose: ROC-AUC chạy từ {sweep_aucs.min():.4f} đến "
      f"{sweep_aucs.max():.4f}.")
print(f"     Tương quan giữa ROC-AUC và ĐỘ PHỦ của vị trí Glucose: r = {r_sweep:+.3f}")
print(f"  4. Trên 10 hoán vị ngẫu nhiên, tương quan tương tự: r = {r_perm:+.3f}")
print()
print("DIỄN GIẢI")
if r_sweep >= 0.5:
    print(f"  Chỉ dịch MỘT cột (Glucose) mà không đụng tới 7 cột còn lại đã đủ làm")
    print(f"  ROC-AUC dao động {sweep_aucs.max() - sweep_aucs.min():.4f}, và mức dao động")
    print(f"  này bám sát ĐỘ PHỦ của vị trí (r = {r_sweep:+.3f}). Vậy cái mà CNN phản")
    print("  ứng là VỊ TRÍ của đặc trưng mạnh trong chuỗi — tức một tạo tác của")
    print("  kiến trúc tích chập 'valid' + pooling — chứ KHÔNG phải quan hệ ngữ")
    print("  nghĩa giữa các cột liền kề. Dữ liệu bảng không có tính cục bộ không")
    print("  gian để khai thác, đúng Cảnh báo Mô hình hoá ở slide 8.")
elif r_sweep <= -0.5:
    print(f"  ROC-AUC tương quan NGHỊCH với độ phủ (r = {r_sweep:+.3f}) — ngược với")
    print("  giả thuyết độ phủ. Cần thêm nghiệm thức để kết luận; xem thảo luận.")
else:
    print(f"  Độ phủ KHÔNG giải thích được biến động (r = {r_sweep:+.3f}). Việc chỉ")
    print(f"  dịch một cột vẫn làm ROC-AUC dao động {sweep_aucs.max() - sweep_aucs.min():.4f},")
    print("  nên thứ tự cột có ảnh hưởng, nhưng qua một cơ chế phức tạp hơn hồ sơ")
    print("  độ phủ đơn thuần (tương tác giữa vị trí, pooling và cặp cột ghép đôi).")
    print("  Điều chắc chắn rút ra được: KHÔNG có bằng chứng nào cho thấy tích chập")
    print("  khai thác quan hệ ngữ nghĩa giữa các cột liền kề — mọi biến động đều")
    print("  là tạo tác kiến trúc, đúng tinh thần Cảnh báo ở slide 8.")
print()
print("  Đối chứng: Logistic Regression cho ROC-AUC không đổi qua mọi hoán vị")
print(f"  (độ lệch chuẩn {lr_aucs.std(ddof=1):.1e}), xác nhận biến động quan sát được")
print("  sinh ra từ KIẾN TRÚC CNN chứ không từ quy trình đánh giá hay dữ liệu.")

QUAN SÁT
  1. Đảo ngược lệch +0.000000 so với gốc.
     -> đúng như dự đoán lý thuyết: đây là phép ĐỐI XỨNG của kiến trúc
        (hồ sơ độ phủ [1, 3, 6, 8, 8, 6, 3, 1] đối xứng qua tâm),
        nên nó KIỂM CHỨNG cài đặt chứ không tách được biến nào.
  2. Hoán vị ngẫu nhiên giảm trung bình +0.0420 (2.4 lần độ lệch chuẩn); 0/10 hoán vị đạt >= gốc.
  3. Quét vị trí Glucose: ROC-AUC chạy từ 0.7967 đến 0.8523.
     Tương quan giữa ROC-AUC và ĐỘ PHỦ của vị trí Glucose: r = -0.233
  4. Trên 10 hoán vị ngẫu nhiên, tương quan tương tự: r = +0.386

DIỄN GIẢI
  Độ phủ KHÔNG giải thích được biến động (r = -0.233). Việc chỉ
  dịch một cột vẫn làm ROC-AUC dao động 0.0556,
  nên thứ tự cột có ảnh hưởng, nhưng qua một cơ chế phức tạp hơn hồ sơ
  độ phủ đơn thuần (tương tác giữa vị trí, pooling và cặp cột ghép đôi).
  Điều chắc chắn rút ra được: KHÔNG có bằng chứng nào cho thấy tích chập
  khai thác quan hệ ngữ nghĩa giữa các cột liền kề — mọi biến động đều
  là tạo tác kiến trúc, đúng tinh thần Cảnh

## 12. Trực quan hoá

### 12.1. Đường cong huấn luyện của ba nền tảng

In [26]:
# ============================================================================
# KHỐI 24 — Hình 3: đường cong huấn luyện, ba nền tảng cạnh nhau
# ============================================================================
fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))

axes[0].plot(cnn.history["train_loss"], label="Train", color="#2563eb", linewidth=1.8)
axes[0].plot(cnn.history["val_loss"], label="Validation", color="#ef4444", linewidth=1.8)
axes[0].set_title("(a) NumPy from scratch", fontweight="bold")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Binary Cross-Entropy")
axes[0].legend()

axes[1].plot(torch_hist["train_loss"], label="Train", color="#2563eb", linewidth=1.8)
axes[1].plot(torch_hist["val_loss"], label="Validation", color="#ef4444", linewidth=1.8)
axes[1].set_title("(b) PyTorch", fontweight="bold")
axes[1].set_xlabel("Epoch")
axes[1].legend()

axes[2].plot(tf_hist.history["loss"], label="Train", color="#2563eb", linewidth=1.8)
axes[2].plot(tf_hist.history["val_loss"], label="Validation", color="#ef4444", linewidth=1.8)
axes[2].set_title("(c) TensorFlow/Keras", fontweight="bold")
axes[2].set_xlabel("Epoch")
axes[2].legend()

for ax in axes:
    ax.set_ylim(bottom=0)
plt.suptitle("Hình 3 — Đường cong học của cùng một CNN trên ba nền tảng",
             fontweight="bold", y=1.03)
plt.tight_layout()
plt.savefig(FIG / "c1_fig3_curves.png", bbox_inches="tight")
plt.show()

print("Số epoch đã chạy: NumPy = %d, PyTorch = %d, TensorFlow = %d"
      % (len(cnn.history["train_loss"]), len(torch_hist["train_loss"]),
         len(tf_hist.history["loss"])))

Số epoch đã chạy: NumPy = 64, PyTorch = 71, TensorFlow = 84


C:\Users\admin\AppData\Local\Temp\ipykernel_23616\3112865517.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 12.2. Ma trận nhầm lẫn và đường cong ROC

In [27]:
# ============================================================================
# KHỐI 25 — Hình 4: ma trận nhầm lẫn (trước/sau hiệu chỉnh ngưỡng) + ROC
# ============================================================================
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))

for ax, thr, title in [
    (axes[0], 0.5, "(a) CNN-5 NumPy — ngưỡng 0.5"),
    (axes[1], best_thr["CNN-5 (NumPy)"],
     f"(b) CNN-5 NumPy — ngưỡng {best_thr['CNN-5 (NumPy)']:.2f}"),
]:
    cm = confusion_matrix(yte, (proba["CNN-5 (NumPy)"] >= thr).astype(int))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
                xticklabels=["Dự đoán 0", "Dự đoán 1"],
                yticklabels=["Thực tế 0", "Thực tế 1"], annot_kws={"size": 14})
    tn, fp, fn, tp = cm.ravel()
    ax.set_title(f"{title}\nFN = {fn} (bỏ sót), FP = {fp} (báo động sai)",
                 fontweight="bold", fontsize=10)
    ax.grid(False)

for name, color in [("CNN-5 (NumPy)", "#2563eb"), ("CNN-5 (PyTorch)", "#16a34a"),
                    ("CNN-5 (TensorFlow)", "#f59e0b"),
                    ("MLP-5 (A03, PyTorch)", "#a855f7"),
                    ("Random Forest", "#64748b")]:
    fpr, tpr, _ = roc_curve(yte, proba[name])
    axes[2].plot(fpr, tpr, color=color, linewidth=1.9,
                 label=f"{name} ({roc_auc_score(yte, proba[name]):.3f})")
axes[2].plot([0, 1], [0, 1], "k--", linewidth=1, alpha=0.5)
axes[2].set_title("(c) Đường cong ROC trên tập test", fontweight="bold")
axes[2].set_xlabel("False Positive Rate")
axes[2].set_ylabel("True Positive Rate")
axes[2].legend(fontsize=7.5, loc="lower right")

plt.suptitle("Hình 4 — Ma trận nhầm lẫn và ROC", fontweight="bold", y=1.03)
plt.tight_layout()
plt.savefig(FIG / "c1_fig4_confusion_roc.png", bbox_inches="tight")
plt.show()

C:\Users\admin\AppData\Local\Temp\ipykernel_23616\3998797910.py:36: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 12.3. Đối sánh mô hình và thí nghiệm hoán vị cột

In [28]:
# ============================================================================
# KHỐI 26 — Hình 5: (a) ROC-AUC mọi mô hình, (b) hoán vị cột, (c) thời gian chạy
# ============================================================================
fig, axes = plt.subplots(1, 3, figsize=(16.5, 4.8))

order_plot = res_df.sort_values("ROC-AUC").index
colors = ["#2563eb" if "CNN" in n else "#a855f7" if "MLP" in n else "#94a3b8"
          for n in order_plot]
vals = res_df.loc[order_plot, "ROC-AUC"].values
axes[0].barh(range(len(order_plot)), vals, color=colors)
axes[0].set_yticks(range(len(order_plot)))
axes[0].set_yticklabels(order_plot, fontsize=8)
for i, v in enumerate(vals):
    axes[0].text(v + 0.004, i, f"{v:.4f}", va="center", fontsize=8, fontweight="bold")
axes[0].set_xlim(min(vals) - 0.04, max(vals) + 0.035)
axes[0].set_title("(a) ROC-AUC trên tập test", fontweight="bold")
axes[0].set_xlabel("ROC-AUC")

ax_b = axes[1]
ax_b.plot(range(8), sweep_aucs, "o-", color="#ef4444", linewidth=2, markersize=7,
          label="ROC-AUC khi dịch Glucose", zorder=3)
ax_b.axhline(base_auc, color="#2563eb", linestyle="--", linewidth=1.3,
             label=f"Thứ tự gốc = {base_auc:.4f}")
ax_b.axhline(perm_aucs.mean(), color="#b45309", linestyle=":", linewidth=1.4,
             label=f"TB hoán vị = {perm_aucs.mean():.4f}")
ax_b.set_xlabel("Vị trí của Glucose trong chuỗi")
ax_b.set_ylabel("ROC-AUC test", color="#ef4444")
ax_b.tick_params(axis="y", labelcolor="#ef4444")
ax_b.set_xticks(range(8))
ax_b.set_title(f"(b) Dịch riêng Glucose — độ phủ vị trí\n(tương quan r = {r_sweep:+.3f})",
               fontweight="bold", fontsize=9.5)
ax_b.legend(fontsize=7, loc="lower right")

ax_b2 = ax_b.twinx()
ax_b2.bar(range(8), COVER, color="#94a3b8", alpha=0.35, width=0.6, zorder=1)
ax_b2.set_ylabel("Độ phủ của vị trí (số đường đi)", color="#475569", fontsize=8.5)
ax_b2.tick_params(axis="y", labelcolor="#475569")
ax_b2.grid(False)
ax_b2.set_ylim(0, COVER.max() * 1.6)

names_t = ["NumPy\nfrom scratch", "PyTorch", "TensorFlow"]
times = [numpy_time, torch_time, tf_time]
epochs_run = [len(cnn.history["train_loss"]), len(torch_hist["train_loss"]),
              len(tf_hist.history["loss"])]
per_ep = [t / e for t, e in zip(times, epochs_run)]
bars = axes[2].bar(names_t, per_ep, color=["#2563eb", "#16a34a", "#f59e0b"], width=0.55)
for bar, t, e in zip(bars, times, epochs_run):
    axes[2].text(bar.get_x() + bar.get_width() / 2, bar.get_height() * 1.02,
                 f"{bar.get_height()*1000:.0f} ms/epoch\n({t:.1f}s / {e} ep)",
                 ha="center", fontsize=8, fontweight="bold")
axes[2].set_title("(c) Thời gian huấn luyện trên mỗi epoch", fontweight="bold")
axes[2].set_ylabel("giây / epoch")
axes[2].set_ylim(0, max(per_ep) * 1.35)

plt.suptitle("Hình 5 — Đối sánh mô hình, hoán vị cột và chi phí huấn luyện",
             fontweight="bold", y=1.03)
plt.tight_layout()
plt.savefig(FIG / "c1_fig5_compare.png", bbox_inches="tight")
plt.show()

C:\Users\admin\AppData\Local\Temp\ipykernel_23616\1859219678.py:59: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 12.4. Mạng đã học được gì? — Trực quan hoá kernel và bản đồ đặc trưng

In [29]:
# ============================================================================
# KHỐI 27 — Hình 6: 16 kernel của tầng Conv1 và bản đồ đặc trưng của một bệnh nhân
# Mỗi kernel dài 3 là một "bộ dò mẫu cục bộ" trên ba đặc trưng liền kề.
# ============================================================================
W1 = cnn.layers[0].W[:, 0, :]        # (16, 3)

fig, axes = plt.subplots(1, 3, figsize=(16.5, 4.6))

im = axes[0].imshow(W1, cmap="RdBu_r", aspect="auto",
                    vmin=-np.abs(W1).max(), vmax=np.abs(W1).max())
axes[0].set_title("(a) 16 kernel của tầng Conv1 (mỗi hàng là một kernel dài 3)",
                  fontweight="bold", fontsize=9.5)
axes[0].set_xlabel("Vị trí trong kernel")
axes[0].set_ylabel("Chỉ số bộ lọc")
axes[0].set_xticks([0, 1, 2])
axes[0].grid(False)
plt.colorbar(im, ax=axes[0], shrink=0.85)

# Bản đồ đặc trưng sau Conv1 + ReLU cho một bệnh nhân dương tính
pos_idx = int(np.argmax(proba["CNN-5 (NumPy)"]))
xs = Xte[pos_idx:pos_idx + 1]
h1 = relu(conv1d_forward(xs, cnn.layers[0].W, cnn.layers[0].b)[0])[0]   # (16, 6)
im2 = axes[1].imshow(h1, cmap="viridis", aspect="auto")
axes[1].set_title(f"(b) ReLU(Conv1) — bệnh nhân rủi ro cao nhất\n"
                  f"(p = {proba['CNN-5 (NumPy)'][pos_idx]:.3f})",
                  fontweight="bold", fontsize=9.5)
axes[1].set_xlabel("Vị trí trên chuỗi (cửa sổ 3 đặc trưng liền kề)")
axes[1].set_ylabel("Chỉ số bộ lọc")
axes[1].grid(False)
plt.colorbar(im2, ax=axes[1], shrink=0.85)

# Mỗi vị trí cửa sổ tương ứng bộ ba đặc trưng nào
win_labels = [f"{FEATURES[i][:4]}·{FEATURES[i+1][:4]}·{FEATURES[i+2][:4]}"
              for i in range(6)]
act_per_pos = h1.sum(axis=0)
axes[2].bar(range(6), act_per_pos, color="#0ea5e9", width=0.62)
axes[2].set_xticks(range(6))
axes[2].set_xticklabels(win_labels, rotation=42, ha="right", fontsize=7)
axes[2].set_title("(c) Tổng kích hoạt theo từng cửa sổ 3 đặc trưng",
                  fontweight="bold", fontsize=9.5)
axes[2].set_ylabel("Σ ReLU(Conv1)")

plt.suptitle("Hình 6 — Bên trong tầng tích chập thứ nhất", fontweight="bold", y=1.03)
plt.tight_layout()
plt.savefig(FIG / "c1_fig6_kernels.png", bbox_inches="tight")
plt.show()

print("Cửa sổ có tổng kích hoạt lớn nhất:", win_labels[int(np.argmax(act_per_pos))])
print("\nLưu ý diễn giải: 'cửa sổ' ở đây chỉ là ba cột liền nhau trong file CSV,")
print("nên nhãn của nó KHÔNG mang ý nghĩa y học. Đó chính là hệ quả của việc")
print("dữ liệu bảng không có tính cục bộ không gian.")

Cửa sổ có tổng kích hoạt lớn nhất: Insu·BMI·Diab

Lưu ý diễn giải: 'cửa sổ' ở đây chỉ là ba cột liền nhau trong file CSV,
nên nhãn của nó KHÔNG mang ý nghĩa y học. Đó chính là hệ quả của việc
dữ liệu bảng không có tính cục bộ không gian.


C:\Users\admin\AppData\Local\Temp\ipykernel_23616\3745248597.py:46: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 13. Xuất mô hình cho web app

Bundle `model_cnn.json` chứa đủ mọi thứ frontend cần để tái lập kết quả: trọng
số từng tầng, tên đặc trưng, median dùng để điền thiếu, mean/scale của scaler,
ngưỡng tối ưu, các chỉ số đánh giá và lịch sử huấn luyện. Nhờ vậy trang web chỉ
cần phép nhân + cộng + ReLU + Sigmoid, **không cần Python và không cần backend**.

In [30]:
# ============================================================================
# KHỐI 28 — Đóng gói và ghi model_cnn.json
# File này được commit lên GitHub; frontend fetch nó rồi suy luận bằng JavaScript
# => deploy serverless trên Vercel (static hosting + tính toán phía client).
# ============================================================================
bundle = cnn.to_dict(decimals=6)
bundle.update({
    "model_name": "CNN 1D 5 tầng (NumPy from scratch)",
    "assignment": "Assignment 04",
    "input_shape": [1, 8],
    "feature_names": FEATURES,
    "class_names": ["Không tiểu đường", "Tiểu đường"],
    "impute_median": [float(medians[c]) if c in ZERO_AS_NAN else None for c in FEATURES],
    "zero_as_nan": ZERO_AS_NAN,
    "scaler": {"mean_": scaler.mean_.tolist(), "scale_": scaler.scale_.tolist()},
    "threshold": best_thr["CNN-5 (NumPy)"],
    "metrics": {k.lower().replace("-", "_"): float(res_df.loc["CNN-5 (NumPy)", k])
                for k in ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]},
    "metrics_tuned": {k.lower().replace("-", "_"): float(tuned_df.loc["CNN-5 (NumPy)", k])
                      for k in ["Accuracy", "Precision", "Recall", "F1"]},
    "comparison": {
        name: {k.lower().replace("-", "_"): float(res_df.loc[name, k])
               for k in ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]}
        for name in res_df.index
    },
    "framework_divergence": divergence.to_dict(orient="records"),
    "permutation_test": {
        "base_auc": float(base_auc),
        "reversed_auc": float(rev_auc),
        "perm_auc_mean": float(perm_aucs.mean()),
        "perm_auc_std": float(perm_aucs.std(ddof=1)),
        "perm_auc_min": float(perm_aucs.min()),
        "perm_auc_max": float(perm_aucs.max()),
        "n_trials": int(len(perm_aucs)),
        "control_logreg_std": float(lr_aucs.std(ddof=1)),
        "coverage_profile": COVER.astype(int).tolist(),
        "glucose_sweep_auc": [round(float(a), 6) for a in sweep_aucs],
        "coverage_corr_sweep": round(r_sweep, 4),
        "coverage_corr_perm": round(r_perm, 4),
    },
    "architecture_text": "8 → Conv(16) → Conv(16) → Pool → Conv(32) → Dense(16) → 1",
    "training": {
        "epochs_run": len(cnn.history["train_loss"]),
        "batch_size": HP["batch_size"], "lr": HP["lr"],
        "l2": HP["l2"], "dropout": HP["dropout"],
        "numpy_seconds": round(numpy_time, 2),
        "pytorch_seconds": round(torch_time, 2),
        "tensorflow_seconds": round(tf_time, 2),
    },
    "history": {
        "train_loss": [round(v, 5) for v in cnn.history["train_loss"]],
        "val_loss": [round(v, 5) for v in cnn.history["val_loss"]],
        "train_acc": [round(v, 5) for v in cnn.history["train_metric"]],
        "val_acc": [round(v, 5) for v in cnn.history["val_metric"]],
    },
})

out_path = ROOT / "model_cnn.json"
out_path.write_text(json.dumps(bundle, ensure_ascii=False, separators=(",", ":")),
                    encoding="utf-8")
print(f"✅ Đã ghi {out_path.name} — {out_path.stat().st_size/1024:.1f} KB")
print(f"   {bundle['n_params']:,} tham số, {bundle['n_trainable_layers']} tầng huấn luyện được")

✅ Đã ghi model_cnn.json — 30.0 KB
   2,449 tham số, 5 tầng huấn luyện được


### 13.1. Kiểm tra parity: NumPy ↔ JSON

Chạy lại thuật toán suy luận **kiểu JavaScript** (`forward_reference`) trên toàn
bộ tập test và so với xác suất do mô hình gốc đưa ra. `assert` bảo đảm sai số
< 1e-5, tức việc làm tròn 6 chữ số khi ghi JSON không làm đổi kết quả — web app
sẽ cho ra đúng con số như notebook.

In [31]:
# ============================================================================
# KHỐI 29 — Parity NumPy <-> JSON trên toàn bộ tập test
# ============================================================================
reloaded = json.loads(out_path.read_text(encoding="utf-8"))
max_err = 0.0
for i in range(len(Xte)):
    ref = forward_reference(reloaded, Xte[i])
    max_err = max(max_err, abs(float(ref) - float(proba["CNN-5 (NumPy)"][i])))
print(f"Sai số tuyệt đối lớn nhất giữa mô hình gốc và bundle JSON: {max_err:.3e}")
assert max_err < 1e-5, "Parity FAILED"
print("✅ Parity PASSED — web app sẽ cho kết quả trùng khớp với notebook.")

Sai số tuyệt đối lớn nhất giữa mô hình gốc và bundle JSON: 2.181e-06
✅ Parity PASSED — web app sẽ cho kết quả trùng khớp với notebook.


In [32]:
# ============================================================================
# KHỐI 30 — Ghi vài mẫu tham chiếu để script JS tự kiểm tra (model_cnn_samples.json)
# ============================================================================
sample_idx = list(range(min(12, len(Xte))))
samples = [{
    "raw": {FEATURES[j]: float(X_test.iloc[i, j]) for j in range(len(FEATURES))},
    "scaled": [round(float(v), 8) for v in Xte[i, 0]],
    "expected_proba": round(float(proba["CNN-5 (NumPy)"][i]), 10),
    "expected_label": int(proba["CNN-5 (NumPy)"][i] >= best_thr["CNN-5 (NumPy)"]),
} for i in sample_idx]
samp_path = ROOT / "model_cnn_samples.json"
samp_path.write_text(json.dumps(samples, ensure_ascii=False, indent=1), encoding="utf-8")
print(f"✅ Đã ghi {samp_path.name} — {len(samples)} mẫu tham chiếu cho kiểm thử JS")

✅ Đã ghi model_cnn_samples.json — 12 mẫu tham chiếu cho kiểm thử JS


## 14. Kết luận Bài toán 1

In [33]:
# ============================================================================
# KHỐI 31 — Tóm tắt số liệu chính
# ============================================================================
cnn_auc = res_df.loc["CNN-5 (NumPy)", "ROC-AUC"]
mlp_auc = res_df.loc["MLP-5 (A03, PyTorch)", "ROC-AUC"]
best_classical = max(classical, key=lambda k: res_df.loc[k, "ROC-AUC"])
bc_auc = res_df.loc[best_classical, "ROC-AUC"]

print("=" * 68)
print("BÀI TOÁN 1 — CHẨN ĐOÁN TIỂU ĐƯỜNG BẰNG CNN 1D")
print("=" * 68)
print(f"Kiến trúc          : {bundle['architecture_text']}")
print(f"Tham số            : {cnn.n_params():,} ({bundle['n_trainable_layers']} tầng huấn luyện được)")
print(f"Gradient check     : sai số tương đối {worst:.2e} — backward viết tay ĐÚNG")
print()
print(f"CNN-5 (NumPy)      : ROC-AUC = {cnn_auc:.4f}")
print(f"CNN-5 (PyTorch)    : ROC-AUC = {res_df.loc['CNN-5 (PyTorch)', 'ROC-AUC']:.4f}")
print(f"CNN-5 (TensorFlow) : ROC-AUC = {res_df.loc['CNN-5 (TensorFlow)', 'ROC-AUC']:.4f}")
print(f"MLP-5 (A03)        : ROC-AUC = {mlp_auc:.4f}")
print(f"{best_classical:<19}: ROC-AUC = {bc_auc:.4f}  (baseline truyền thống tốt nhất)")
print()
print(f"CNN so với MLP-5   : {cnn_auc - mlp_auc:+.4f}")
print(f"CNN so với {best_classical:<9}: {cnn_auc - bc_auc:+.4f}")
print()
print("Thời gian huấn luyện (giây/epoch):")
for nm, t, e in zip(["NumPy", "PyTorch", "TensorFlow"], times, epochs_run):
    print(f"  {nm:<12}: {t/e*1000:7.1f} ms/epoch  (tổng {t:.1f}s, {e} epoch)")
print()
print("Thí nghiệm thứ tự cột:")
print(f"  gốc             = {base_auc:.4f}")
print(f"  đảo ngược       = {rev_auc:.4f}  (phép đối xứng của kiến trúc, chênh {rev_auc-base_auc:+.1e})")
print(f"  hoán vị (n={N_PERM})  = {perm_aucs.mean():.4f} ± {perm_aucs.std(ddof=1):.4f}")
print(f"  quét Glucose    = [{sweep_aucs.min():.4f}, {sweep_aucs.max():.4f}], "
      f"tương quan với độ phủ r = {r_sweep:+.3f}")
print("  => chỉ dịch MỘT cột đã làm hiệu năng dao động, nên thứ tự cột chỉ ảnh")
print("     hưởng như một TẠO TÁC KIẾN TRÚC; không có bằng chứng nào cho thấy")
print("     tích chập khai thác quan hệ ngữ nghĩa giữa các cột (đúng slide 8).")
print("=" * 68)

BÀI TOÁN 1 — CHẨN ĐOÁN TIỂU ĐƯỜNG BẰNG CNN 1D
Kiến trúc          : 8 → Conv(16) → Conv(16) → Pool → Conv(32) → Dense(16) → 1
Tham số            : 2,449 (5 tầng huấn luyện được)
Gradient check     : sai số tương đối 4.23e-07 — backward viết tay ĐÚNG

CNN-5 (NumPy)      : ROC-AUC = 0.8355
CNN-5 (PyTorch)    : ROC-AUC = 0.8336
CNN-5 (TensorFlow) : ROC-AUC = 0.8368
MLP-5 (A03)        : ROC-AUC = 0.8391
Logistic Regression: ROC-AUC = 0.8345  (baseline truyền thống tốt nhất)

CNN so với MLP-5   : -0.0036
CNN so với Logistic Regression: +0.0010

Thời gian huấn luyện (giây/epoch):
  NumPy       :    32.2 ms/epoch  (tổng 2.1s, 64 epoch)
  PyTorch     :    99.1 ms/epoch  (tổng 7.0s, 71 epoch)
  TensorFlow  :   200.0 ms/epoch  (tổng 16.8s, 84 epoch)

Thí nghiệm thứ tự cột:
  gốc             = 0.8355
  đảo ngược       = 0.8355  (phép đối xứng của kiến trúc, chênh +1.1e-16)
  hoán vị (n=10)  = 0.7936 ± 0.0172
  quét Glucose    = [0.7967, 0.8523], tương quan với độ phủ r = -0.233
  => chỉ dịch MỘT c

### Nhận xét trung thực

1. **Bản tự cài là đúng.** Tích chập im2col khớp tuyệt đối với bản viết theo
   công thức toán học; gradient viết tay khớp gradient sai phân số ở mức
   $10^{-8}$. Ba nền tảng cho cùng số tham số và ROC-AUC sát nhau.

2. **CNN không vượt trội trên dữ liệu bảng.** Ba bản CNN đạt ROC-AUC quanh
   0,83–0,84, tức **ngang** MLP-5 của Assignment 03 và ngang Logistic
   Regression, dù dùng ít tham số hơn MLP gần 5 lần. Không có cải thiện nào
   đến từ tích chập.

3. **Thứ tự cột ảnh hưởng mạnh — và đó là một tạo tác, không phải tri thức.**
   Đây là kết quả đáng chú ý nhất của bài, và nó đến từ việc thiết kế thí
   nghiệm cẩn thận thay vì chỉ "xáo trộn rồi xem".

   - **Đảo ngược cho ROC-AUC trùng khít tới $10^{-16}$.** Điều này được *dự
     đoán trước* chứ không phải phát hiện tình cờ: hồ sơ độ phủ
     $[1,3,6,8,8,6,3,1]$ đối xứng qua tâm, nên đảo ngược là một **phép đối
     xứng của kiến trúc**. Quan sát khớp dự đoán là một phép kiểm chứng độc
     lập rằng cài đặt đúng — nhưng nó **không** nói gì về tính cục bộ.
   - **Chỉ dịch riêng `Glucose` qua 8 vị trí, không đụng tới 7 cột còn lại, đã
     làm ROC-AUC dao động 0,056** — rộng hơn cả khoảng biến động của hoán vị
     ngẫu nhiên. Và mức dao động này **không** bám theo độ phủ vị trí
     ($r = -0{,}23$), tức không có quy luật diễn giải được nào.
   - Logistic Regression, bất biến *chính xác* với hoán vị (độ lệch chuẩn
     $10^{-16}$), xác nhận toàn bộ biến động do **kiến trúc CNN** sinh ra chứ
     không do dữ liệu hay quy trình đánh giá.

   **Hệ quả đáng lo và đáng nhớ nhất:** biến động do một lựa chọn *tuỳ tiện*
   (đặt cột nào ở đâu trong file CSV) lên tới **0,056**, trong khi toàn bộ
   chênh lệch giữa CNN và các baseline chỉ cỡ **0,001–0,004**. Nói cách khác,
   **nhiễu kiến trúc lớn gấp hơn 10 lần tín hiệu cần đo**. Bất kỳ kết luận nào
   kiểu "CNN tốt hơn/kém hơn mô hình X trên dữ liệu bảng" mà không kiểm soát
   thứ tự cột đều không có cơ sở.

   Tóm lại: tích chập ở đây hoạt động như một **cách chia sẻ tham số nhạy cảm
   với vị trí**, chứ không phải bộ dò mẫu cục bộ có ý nghĩa — đúng Cảnh báo Mô
   hình hoá ở slide 8.

4. **Ba nền tảng không trùng khít từng chữ số** — và đó là điều bình thường.
   Nguyên nhân là thứ tự cộng dồn dấu phẩy động, bộ sinh số giả ngẫu nhiên và
   thứ tự trộn mini-batch khác nhau, chứ không phải một bản nào sai.

5. **Bản NumPy lại chạy nhanh nhất ở quy mô này.** Điều nghe có vẻ nghịch lý
   này đến từ chi phí cố định: mô hình chỉ có 2.449 tham số và mỗi batch chỉ 32
   mẫu, nên thời gian bị chi phối bởi *overhead* mỗi bước (dựng graph, đồng bộ
   tensor, gọi kernel) chứ không bởi số phép nhân. Ở quy mô lớn — như Bài toán
   3 với ma trận nhúng — trật tự này đảo lại.

6. **Giá trị sư phạm nằm ở chỗ khác.** Bài này cho thấy rõ cái *không đổi*
   (toán học, dữ liệu, cách đánh giá) và cái *đổi* (autograd, optimizer, cú
   pháp) khi chuyển giữa ba nền tảng — đúng thông điệp của Lecture 04. Và nó
   cho một bài học phương pháp: **một phép thử can thiệp hai biến cùng lúc
   không kết luận được gì**; phải thêm nghiệm thức tách biến (đảo ngược) mới
   đọc đúng nguyên nhân.